# Evo+Swarm (GA+PSO) Sea Route Planner — Kaohsiung → Sihanoukville

This notebook computes the shortest **sea-only** route (geodesic length) between **Kaohsiung** and **Sihanoukville**, avoiding land using a land mask.  
It uses a **hybrid Evolutionary (GA) + Swarm (PSO)** approach:

- **GA (outer loop)**: explores coarse route shapes (waypoint positions)
- **PSO (inner loop)**: locally refines continuous waypoint coordinates for each GA individual

**Inputs**:
- Land mask: Natural Earth 10m land (**shapefile**, WGS84)
- Optional corridor (generated from `scgraph` maritime lines)

**Outputs**:
- GeoJSON route, CSV route
- Folium HTML map (auto-open with `webbrowser` when run locally)

## 0. Configuration

In [14]:
# =========================
# USER CONFIGURATION
# =========================

# 1) Land shapefile (Natural Earth 10m)
#    >>> Set to your local path <<<
LAND_SHP = r"C:\Users\slab\Desktop\Slab Project\Stage1\data\Land\ne_10m_land.shp"

# 2) Start & End (Kaohsiung, Sihanoukville) — can adjust later

START_LON, START_LAT = 120.27, 22.62
END_LON,   END_LAT   = 103.52958, 10.60932

# 3) Derived AOI: pad the bbox of start/end by ±2 degrees
AOI_PADDING_DEG = 3.0

# 4) Corridor parameters
USE_SCGRAPH_CORRIDOR = False     # if scgraph import fails, will gracefully fallback
CORRIDOR_HALF_WIDTH_M = 3000.0   # buffer half-width for corridor polygon (~2 km)
CORRIDOR_PENALTY = 0.02          # lambda for small corridor "attraction" penalty (normalized)

# 5) Land safety buffer
LAND_BUFFER_M = 500.0            # 0.5 km (as requested)

# 6) Waypoints (excluding start/end)
K_WAYPOINTS = 3

# 7) Evo+Swarm parameters
RANDOM_SEED = 42

# GA parameters
GA_POP = 20
GA_GENS = 30
GA_CROSSOVER_RATE = 0.8
GA_MUTATION_RATE = 0.2
GA_ELITISM = 2

# PSO parameters (per GA individual)
PSO_PARTICLES = 8
PSO_ITERS = 10
PSO_W_START = 0.9
PSO_W_END = 0.4
PSO_C1 = 1.6
PSO_C2 = 1.6

# 8) Outputs
OUTPUT_DIR = "outputs"
HTML_NAME = "route_kaohsiung_sihanoukville.html"
GEOJSON_NAME = "route_kaohsiung_sihanoukville.geojson"
CSV_NAME = "route_kaohsiung_sihanoukville.csv"

# Ensure output dir exists
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Config loaded.")

Config loaded.


## 1. Imports & Utilities

In [15]:
import sys, os, math, random, json, webbrowser, warnings, itertools
from typing import List, Tuple, Optional
import numpy as np

# Geo stack
import geopandas as gpd
from shapely.geometry import LineString, MultiLineString, Point, Polygon, box
from shapely.ops import nearest_points
from shapely import STRtree

# Projections & geodesic
from pyproj import Geod, CRS, Transformer

# Optional: geographiclib as an alternative geodesic backend
try:
    from geographiclib.geodesic import Geodesic
    GEO_LIB = Geodesic.WGS84
except Exception:
    GEO_LIB = None

# Folium map
import folium

warnings.filterwarnings("ignore")

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Geodesic distance (km) between 2 lon/lat points
GEOD = Geod(ellps="WGS84")

def geodesic_km(a: Tuple[float,float], b: Tuple[float,float]) -> float:
    lon1, lat1 = a
    lon2, lat2 = b
    if GEO_LIB is not None:
        inv = GEO_LIB.Inverse(lat1, lon1, lat2, lon2)
        return inv["s12"] / 1000.0
    # fallback
    _, _, dist_m = GEOD.inv(lon1, lat1, lon2, lat2)
    return dist_m / 1000.0

def polyline_length_km(coords: List[Tuple[float,float]]) -> float:
    return sum(geodesic_km(coords[i], coords[i+1]) for i in range(len(coords)-1))

def compute_aoi(start, end, pad=2.0):
    x0 = min(start[0], end[0]) - pad
    y0 = min(start[1], end[1]) - pad
    x1 = max(start[0], end[0]) + pad
    y1 = max(start[1], end[1]) + pad
    return (x0, y0, x1, y1)

def ensure_crs4326(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    if gdf.crs is None:
        gdf = gdf.set_crs(4326)
    else:
        gdf = gdf.to_crs(4326)
    return gdf

def project_local_m(gdf: gpd.GeoDataFrame, center_lonlat: Tuple[float,float]) -> Tuple[gpd.GeoDataFrame, CRS, CRS, Transformer, Transformer]:
    """
    Project to a local metric CRS (EPSG:3857 as a simple approx for buffering small distances).
    Returns projected gdf and transformers.
    """
    crs_wgs = CRS.from_epsg(4326)
    crs_m = CRS.from_epsg(3857)  # Web Mercator (meters) — ok for small buffers
    to_m = Transformer.from_crs(crs_wgs, crs_m, always_xy=True)
    to_wgs = Transformer.from_crs(crs_m, crs_wgs, always_xy=True)
    gdf_m = gdf.to_crs(crs_m)
    return gdf_m, crs_wgs, crs_m, to_m, to_wgs

def to_linestring(coords: List[Tuple[float,float]]) -> LineString:
    return LineString(coords)

def clamp_lonlat(lon, lat):
    return max(-180, min(180, lon)), max(-90, min(90, lat))

print("Imports OK.")

Imports OK.


## 2. Load Land Mask + Buffer (0.5 km) & Build AOI

In [16]:
start = (START_LON, START_LAT)
end   = (END_LON,   END_LAT)

AOI = compute_aoi(start, end, AOI_PADDING_DEG)
print("AOI:", AOI)

land = gpd.read_file(LAND_SHP)
land = ensure_crs4326(land)

# Clip to AOI for speed
aoi_poly = gpd.GeoDataFrame(geometry=[box(*AOI)], crs=4326)
try:
    land = land.clip(aoi_poly)
except Exception:
    land = gpd.overlay(land, aoi_poly, how="intersection")

# Buffer in metric CRS
land_m, crs_wgs, crs_m, to_m, to_wgs = project_local_m(land, ((START_LON+END_LON)/2, (START_LAT+END_LAT)/2))
land_buf_m = land_m.buffer(LAND_BUFFER_M)
land_buf = gpd.GeoDataFrame(geometry=land_buf_m, crs=crs_m).to_crs(4326)

# Build STRtree for fast intersection checks — explode to pure polygons first
land_exploded = land_buf.explode(index_parts=False, ignore_index=True)
land_exploded = land_exploded[
    land_exploded.geometry.notnull() &
    (~land_exploded.geometry.is_empty)
]

from shapely.geometry import Polygon as _Poly, MultiPolygon as _MPoly
land_polys = []
for geom in land_exploded.geometry:
    if isinstance(geom, (_Poly, _MPoly)):
        land_polys.append(geom)

land_tree = STRtree(land_polys)
print(f"Land polygons after buffer & explode: {len(land_polys)}")




AOI: (100.52958, 7.60932, 123.27, 25.62)
Land polygons after buffer & explode: 295


## 3. Corridor from `scgraph` maritime lines (optional)

In [17]:
# Try importing scgraph and reconstruct maritime line segments into a corridor.
corridor_centerline = None
corridor_polygon = None

def segments_to_multiline(segs):
    ls = []
    for s in segs:
        try:
            if isinstance(s, np.ndarray) and s.shape == (2,2):
                a = (float(s[0,0]), float(s[0,1]))
                b = (float(s[1,0]), float(s[1,1]))
                ls.append(LineString([a,b]))
        except Exception:
            continue
    if not ls:
        return None
    return MultiLineString(ls)

if USE_SCGRAPH_CORRIDOR:
    try:
        # Adapted from your snippet
        import numpy as _np
        from scgraph.geographs.marnet import marnet_geograph as MNET
        import random as _random

        def _segments_from_edges_list(edges, nodes_lookup=None):
            segs = []
            for e in edges:
                try:
                    if isinstance(e, (list, tuple)) and len(e) >= 2:
                        u, v = e[0], e[1]
                        if nodes_lookup and u in nodes_lookup and v in nodes_lookup:
                            pu, pv = nodes_lookup[u], nodes_lookup[v]
                            segs.append(_np.asarray([pu, pv], dtype=float))
                    elif isinstance(e, dict):
                        if "geometry" in e and hasattr(e["geometry"], "coords"):
                            coords = _np.asarray(e["geometry"].coords, dtype=float)
                            segs.extend([coords[i:i+2] for i in range(len(coords)-1)])
                        elif "coordinates" in e and isinstance(e["coordinates"], (list, tuple)):
                            coords = _np.asarray(e["coordinates"], dtype=float)
                            if coords.ndim == 2 and coords.shape[1] == 2:
                                segs.extend([coords[i:i+2] for i in range(len(coords)-1)])
                except Exception:
                    continue
            return segs if segs else None

        def _try_graph_like_and_build_segments():
            for attr in ("graph", "_graph", "geograph", "network", "G", "_G"):
                G = getattr(MNET, attr, None)
                if G is None: 
                    continue
                # nodes
                nodes = None
                for nattr in ("nodes", "node", "vertices", "_nodes"):
                    try:
                        obj = getattr(G, nattr)
                    except Exception:
                        obj = None
                    if obj is None:
                        continue
                    try:
                        tmp = {}
                        it = obj(data=True) if callable(obj) else obj
                        for item in it:
                            if isinstance(item, tuple) and len(item) == 2:
                                nid, data = item
                                lon = data.get("longitude") or data.get("lon") or data.get("x")
                                lat = data.get("latitude")  or data.get("lat")  or data.get("y")
                                if lon is not None and lat is not None:
                                    tmp[nid] = (float(lon), float(lat))
                        if tmp:
                            nodes = tmp
                            break
                    except Exception:
                        try:
                            tmp = {}
                            for nid, data in obj.items():
                                if isinstance(data, dict):
                                    lon = data.get("longitude") or data.get("lon") or data.get("x")
                                    lat = data.get("latitude")  or data.get("lat")  or data.get("y")
                                    if lon is not None and lat is not None:
                                        tmp[nid] = (float(lon), float(lat))
                            if tmp:
                                nodes = tmp
                                break
                        except Exception:
                            pass
                # edges
                for eattr in ("edges", "_edges", "links"):
                    edges = getattr(G, eattr, None)
                    if edges is None:
                        continue
                    try:
                        edges = edges() if callable(edges) else edges
                    except Exception:
                        pass
                    segs = _segments_from_edges_list(edges, nodes_lookup=nodes)
                    if segs:
                        return segs
            return None

        def _fallback_segments_by_sampling(aoi, n_paths=40):
            from scgraph.geographs.marnet import marnet_geograph
            segs = []
            if aoi is None:
                aoi = (60, -20, 150, 40)
            x0, y0, x1, y1 = aoi
            nx, ny = 6, 5
            xs = _np.linspace(x0, x1, nx)
            ys = _np.linspace(y0, y1, ny)
            pts = [(float(x), float(y)) for x in xs for y in ys]
            for _ in range(n_paths):
                (lon1, lat1), (lon2, lat2) = _random.sample(pts, 2)
                try:
                    out = marnet_geograph.get_shortest_path(
                        origin_node={"longitude": lon1, "latitude": lat1},
                        destination_node={"longitude": lon2, "latitude": lat2},
                        output_units="km",
                    )
                    path = out.get("coordinate_path", [])
                    coords = []
                    for p in path:
                        if isinstance(p, dict):
                            lon = p.get("longitude") or p.get("lon") or p.get("x")
                            lat = p.get("latitude")  or p.get("lat")  or p.get("y")
                            if lon is not None and lat is not None:
                                coords.append((float(lon), float(lat)))
                        elif isinstance(p, (list, tuple)) and len(p) >= 2:
                            a, b = p[0], p[1]
                            if -180 <= a <= 180 and -90 <= b <= 90:
                                coords.append((float(a), float(b)))
                            elif -90 <= a <= 90 and -180 <= b <= 180:
                                coords.append((float(b), float(a)))
                    if len(coords) >= 2:
                        for i in range(len(coords)-1):
                            segs.append(_np.asarray([coords[i], coords[i+1]], dtype=float))
                except Exception:
                    continue
            return segs

        segs = _try_graph_like_and_build_segments()
        if not segs:
            segs = _fallback_segments_by_sampling(AOI)

        # Filter by AOI
        def in_aoi(pt, aoi):
            x, y = pt
            x0, y0, x1, y1 = aoi
            return (x0 <= x <= x1) and (y0 <= y <= y1)

        segs2 = []
        for s in segs:
            if s is None or len(s) != 2: continue
            if in_aoi(s[0], AOI) or in_aoi(s[1], AOI):
                segs2.append(s)

        ml = segments_to_multiline(segs2)
        if ml is not None:
            # Build corridor polygon by buffering in meters
            #gdf_ml = gpd.GeoDataFrame(geometry=[ml], crs=4326)
            gdf_ml = gpd.GeoDataFrame(geometry=[ml], crs=4326)
            gdf_ml_m = gdf_ml.to_crs(3857)
            corridor_polygon_m = gdf_ml_m.buffer(CORRIDOR_HALF_WIDTH_M)

            # 炸成單一面，避免後面 nearest_points 對複合幾何報型別
            corridor_gdf = gpd.GeoDataFrame(geometry=corridor_polygon_m, crs=3857).to_crs(4326)
            corridor_gdf = corridor_gdf.explode(index_parts=False, ignore_index=True)
            corridor_gdf = corridor_gdf[corridor_gdf.geometry.notnull() & (~corridor_gdf.geometry.is_empty)]
            corridor_polygon = corridor_gdf.unary_union
            corridor_centerline = ml

            print("Corridor constructed from scgraph lines.")
        else:
            print("No maritime lines found for corridor; proceeding without corridor.")
            corridor_centerline = None
            corridor_polygon = None

    except Exception as e:
        print("scgraph import or parsing failed; proceeding without corridor. Error:", e)
        corridor_centerline = None
        corridor_polygon = None
else:
    print("Corridor generation disabled by config.")

Corridor generation disabled by config.


## 4. Feasibility, Repair & Fitness

In [18]:
from shapely.geometry import Polygon as _Poly, MultiPolygon as _MPoly, LineString as _LS

def _as_poly(obj):
    """某些環境下 STRtree.query 可能回傳索引整數，這裡把它映回 land_polys；否則原樣回傳。"""
    if isinstance(obj, (int, np.integer)) and 0 <= int(obj) < len(land_polys):
        return land_polys[int(obj)]
    return obj

# --- Densify a geodesic segment into polyline points every ~max_step_km ---
def densify_geodesic(a: Tuple[float,float], b: Tuple[float,float], max_step_km: float = 10.0):
    """
    Return a list of lon/lat points [a, ..., b] that approximates the geodesic
    between a and b with ~max_step_km spacing (endpoints included).
    """
    total = geodesic_km(a, b)
    # number of interior points (pyproj.Geod.npts excludes endpoints)
    n = max(1, int(total / max_step_km)) - 1
    if n <= 0:
        return [a, b]
    inter = GEOD.npts(a[0], a[1], b[0], b[1], n)  # [(lon,lat), ...] interior only
    return [a] + [(lon, lat) for (lon, lat) in inter] + [b]

# --- Land intersection (segment) ---
def segment_crosses_land(p0, p1) -> bool:
    pts = densify_geodesic(p0, p1, max_step_km=10.0)
    line = LineString(pts)
    candidates = land_tree.query(line)
    for c in candidates:
        poly = _as_poly(c)
        if not isinstance(poly, (_Poly, _MPoly)):
            continue
        if line.intersects(poly):
            return True
    return False

def path_crosses_land(coords: List[Tuple[float,float]]) -> bool:
    for i in range(len(coords)-1):
        if segment_crosses_land(coords[i], coords[i+1]):
            return True
    return False

# --- Simple sea projection (if a point lies on land, push to nearest sea) ---
def project_point_to_sea(pt):
    p = Point(pt)
    candidates = land_tree.query(p)
    hit = None
    for c in candidates:
        poly = _as_poly(c)
        if not isinstance(poly, (_Poly, _MPoly)):
            continue
        if p.within(poly):
            hit = poly
            break
    if hit is None:
        return pt
    # MultiPolygon 用 boundary、Polygon 用 exterior
    boundary = hit.boundary if isinstance(hit, _MPoly) else hit.exterior
    np1, np2 = nearest_points(p, boundary)
    return (float(np2.x), float(np2.y))


# --- Corridor projection & distance ---
def project_point_to_corridor(pt: Tuple[float,float]) -> Tuple[float,float]:
    if corridor_polygon is None:
        return pt
    p = Point(pt)
    if corridor_polygon.contains(p):
        return pt
    # project to nearest boundary if outside
    np1, np2 = nearest_points(p, corridor_polygon)
    return (float(np2.x), float(np2.y))

def corridor_offset_metric(coords: List[Tuple[float,float]]) -> float:
    if corridor_centerline is None:
        return 0.0
    # sample every vertex; compute mean distance (km) to centerline
    dists_km = []
    for c in coords:
        p = Point(c)
        # distance in degrees; convert via local scale by geodesic to a nearby projected point
        # We'll compute nearest point along centerline, then geodesic_km
        try:
            np1, np2 = nearest_points(p, corridor_centerline)
            dists_km.append(geodesic_km((p.x, p.y), (np2.x, np2.y)))
        except Exception:
            dists_km.append(0.0)
    if not dists_km:
        return 0.0
    # Normalize by total length to keep penalty small compared to length
    length_km = polyline_length_km(coords)
    if length_km <= 1e-6:
        return 0.0
    return (sum(dists_km) / len(dists_km)) / max(1.0, length_km)

# --- Fitness ---
BIG_PENALTY_KM = 1000.0  # add if land cross

def fitness(coords: List[Tuple[float,float]]) -> float:
    L = polyline_length_km(coords)
    land_cross = path_crosses_land(coords)
    Dcorr = corridor_offset_metric(coords) if corridor_centerline is not None else 0.0
    score = -L - (CORRIDOR_PENALTY * Dcorr)
    if land_cross:
        score -= BIG_PENALTY_KM
    return score

# --- Snap start/end to sea if they fall onto buffered land ---
start = project_point_to_sea(start)
end   = project_point_to_sea(end)
print("Start (snapped if needed):", start)
print("End   (snapped if needed):", end)

Start (snapped if needed): (120.26631100057529, 22.615675766290686)
End   (snapped if needed): (103.52571861824896, 10.604164604011338)


## 5. Initialization (great-circle skeleton + small perturbations + sea & corridor repair)

In [19]:
def linspace_route(a, b, k):
    """Linear (lon/lat) interpolation to get k waypoints between a and b (coarse)."""
    xs = np.linspace(a[0], b[0], k+2)[1:-1]
    ys = np.linspace(a[1], b[1], k+2)[1:-1]
    return list(zip(xs, ys))

def random_perturb(pt, lon_scale=0.2, lat_scale=0.2):
    """Add small random delta in degrees (tuned for AOI)."""
    return (pt[0] + np.random.normal(0, lon_scale),
            pt[1] + np.random.normal(0, lat_scale))

def init_individual(k: int) -> List[Tuple[float,float]]:
    # base skeleton
    mids = linspace_route(start, end, k)
    # apply small perturbations (scaled by AOI size)
    x0,y0,x1,y1 = AOI
    lon_scale = (x1-x0) * 0.01   # 1% of width
    lat_scale = (y1-y0) * 0.01   # 1% of height
    mids = [random_perturb(pt, lon_scale, lat_scale) for pt in mids]

    # optionally snap one random waypoint to corridor
    if corridor_centerline is not None:
        j = np.random.randint(0, len(mids))
        mids[j] = project_point_to_corridor(mids[j])

    # ensure points at sea
    mids = [project_point_to_sea(pt) for pt in mids]

    # assemble full route
    coords = [start] + mids + [end]

    # if still infeasible, try a quick repair pass: move each interior point slightly toward sea/corridor
    if path_crosses_land(coords):
        mids2 = []
        for pt in mids:
            pt = project_point_to_sea(pt)
            if corridor_centerline is not None:
                pt = project_point_to_corridor(pt)
            mids2.append(pt)
        coords = [start] + mids2 + [end]

    return coords

print("Init helpers ready.")

Init helpers ready.


## 6. GA (outer) + PSO (inner)

In [20]:
# --- PSO local refine for a given route (only optimizes interior waypoints) ---
def pso_refine(coords: List[Tuple[float,float]], iters=PSO_ITERS, particles=PSO_PARTICLES):
    # Flatten interior waypoints to vector x = [lon1,lat1, ..., lonk,latk]
    interior = coords[1:-1]
    x0 = np.array([v for pt in interior for v in pt], dtype=float)

    # bounds from AOI
    x_min = []
    x_max = []
    for i in range(len(x0)//2):
        x_min += [AOI[0], AOI[1]]
        x_max += [AOI[2], AOI[3]]
    x_min = np.array(x_min); x_max = np.array(x_max)

    def vec_to_coords(vec):
        mids = [(vec[2*i], vec[2*i+1]) for i in range(len(vec)//2)]
        return [coords[0]] + mids + [coords[-1]]

    def clamp_vec(vec):
        return np.clip(vec, x_min, x_max)

    # Initialize swarm
    rng = np.random.default_rng(RANDOM_SEED)
    swarm_x = np.tile(x0, (particles,1)) + rng.normal(0, 0.01*(x_max-x_min), size=(particles, len(x0)))
    swarm_v = rng.normal(0, 0.005*(x_max-x_min), size=(particles, len(x0)))

    pbest_x = swarm_x.copy()
    pbest_f = np.array([fitness(vec_to_coords(xx)) for xx in pbest_x])

    # lbest ring topology
    idxs = np.arange(particles)
    left = (idxs - 1) % particles
    right = (idxs + 1) % particles

    def inertia_weight(t):
        return PSO_W_START + (PSO_W_END - PSO_W_START) * (t / max(1, iters-1))

    for t in range(iters):
        # compute local best per particle
        lbest = []
        for i in range(particles):
            neighbors = [i, left[i], right[i]]
            j = neighbors[np.argmax(pbest_f[neighbors])]
            lbest.append(pbest_x[j])
        lbest = np.array(lbest)

        w = inertia_weight(t)
        r1 = rng.random(size=swarm_x.shape)
        r2 = rng.random(size=swarm_x.shape)

        swarm_v = (w*swarm_v
                   + PSO_C1*r1*(pbest_x - swarm_x)
                   + PSO_C2*r2*(lbest - swarm_x))
        swarm_x = clamp_vec(swarm_x + swarm_v)

        # Repair: project any interior point that fell on land/corridor issues
        for i in range(particles):
            mids = [(swarm_x[i,2*j], swarm_x[i,2*j+1]) for j in range(len(x0)//2)]
            mids2 = []
            for pt in mids:
                pt = project_point_to_sea(pt)
                if corridor_centerline is not None:
                    pt = project_point_to_corridor(pt)
                mids2.append(pt)
            # write back
            for j,pt in enumerate(mids2):
                swarm_x[i,2*j] = pt[0]
                swarm_x[i,2*j+1] = pt[1]

        # evaluate
        fvals = np.array([fitness(vec_to_coords(xx)) for xx in swarm_x])

        # update pbest
        improved = fvals > pbest_f
        pbest_x[improved] = swarm_x[improved]
        pbest_f[improved] = fvals[improved]

    # return best
    j = int(np.argmax(pbest_f))
    return vec_to_coords(pbest_x[j])

# --- GA outer loop ---
def tournament_select(pop, fits, k=2):
    idxs = np.random.choice(len(pop), size=k, replace=False)
    best = idxs[0]
    for i in idxs[1:]:
        if fits[i] > fits[best]:
            best = i
    return pop[best]

def crossover(parent1, parent2):
    # two strategies: splice OR blend (choose randomly)
    r = np.random.rand()
    p1_mid = parent1[1:-1]; p2_mid = parent2[1:-1]
    if r < 0.5:
        # splice at a random cut
        cut = np.random.randint(1, len(p1_mid))
        child_mid = p1_mid[:cut] + p2_mid[cut:]
    else:
        # blend each waypoint (BLX-like)
        child_mid = []
        for (a,b) in zip(p1_mid, p2_mid):
            alpha = np.random.uniform(-0.2, 1.2)  # small extrapolation
            cx = a[0] + alpha*(b[0]-a[0])
            cy = a[1] + alpha*(b[1]-a[1])
            child_mid.append((cx, cy))
    # repair to sea/corridor
    child_mid = [project_point_to_sea(pt) for pt in child_mid]
    if corridor_centerline is not None:
        child_mid = [project_point_to_corridor(pt) for pt in child_mid]
    return [parent1[0]] + child_mid + [parent1[-1]]

def mutate(ind, rate=GA_MUTATION_RATE):
    mids = ind[1:-1]
    x0,y0,x1,y1 = AOI
    lon_scale = (x1-x0) * 0.01
    lat_scale = (y1-y0) * 0.01
    mids2 = []
    for pt in mids:
        if np.random.rand() < rate:
            pt = (pt[0] + np.random.normal(0, lon_scale),
                  pt[1] + np.random.normal(0, lat_scale))
        pt = project_point_to_sea(pt)
        if corridor_centerline is not None:
            pt = project_point_to_corridor(pt)
        mids2.append(pt)
    return [ind[0]] + mids2 + [ind[-1]]

def evo_swarm_optimize():
    # init population
    pop = [init_individual(K_WAYPOINTS) for _ in range(GA_POP)]
    fits = np.array([fitness(ind) for ind in pop])

    best_route = pop[int(np.argmax(fits))]
    best_fit = float(np.max(fits))

    history = [(-best_fit, best_fit)]  # store (distance_km, fitness)

    for g in range(GA_GENS):
        new_pop = []

        # elitism
        elite_idx = np.argsort(-fits)[:GA_ELITISM]
        for i in elite_idx:
            new_pop.append(pop[i])

        # fill rest
        while len(new_pop) < GA_POP:
            p1 = tournament_select(pop, fits, k=3)
            p2 = tournament_select(pop, fits, k=3)

            child = p1
            if np.random.rand() < GA_CROSSOVER_RATE:
                child = crossover(p1, p2)
            child = mutate(child, rate=GA_MUTATION_RATE)

            # Local PSO refine
            child = pso_refine(child, iters=PSO_ITERS, particles=PSO_PARTICLES)

            new_pop.append(child)

        pop = new_pop
        fits = np.array([fitness(ind) for ind in pop])

        # track best
        idx = int(np.argmax(fits))
        if fits[idx] > best_fit:
            best_fit = float(fits[idx])
            best_route = pop[idx]

        if (g+1) % 10 == 0 or g == 0:
            dist_km = polyline_length_km(best_route)
            print(f"Gen {g+1:3d} | best distance: {dist_km:.2f} km")
            history.append((dist_km, best_fit))

    return best_route, history

## 7. Run Optimization

In [21]:
best_route, hist = evo_swarm_optimize()
best_dist = polyline_length_km(best_route)
print(f"\nBest distance = {best_dist:.2f} km")

# Save GeoJSON & CSV
geo = {
    "type": "FeatureCollection",
    "features": [{
        "type": "Feature",
        "properties": {"name": "Kaohsiung→Sihanoukville", "distance_km": best_dist},
        "geometry": {
            "type": "LineString",
            "coordinates": [[lon,lat] for (lon,lat) in best_route]
        }
    }]
}
geo_path = os.path.join(OUTPUT_DIR, GEOJSON_NAME)
with open(geo_path, "w", encoding="utf-8") as f:
    json.dump(geo, f, ensure_ascii=False, indent=2)

csv_path = os.path.join(OUTPUT_DIR, CSV_NAME)
with open(csv_path, "w", encoding="utf-8") as f:
    f.write("lon,lat\n")
    for lon,lat in best_route:
        f.write(f"{lon},{lat}\n")

print("Saved:", geo_path)
print("Saved:", csv_path)

Gen   1 | best distance: 2222.94 km
Gen  10 | best distance: 2222.87 km
Gen  20 | best distance: 2222.87 km
Gen  30 | best distance: 2222.87 km

Best distance = 2222.87 km
Saved: outputs\route_kaohsiung_sihanoukville.geojson
Saved: outputs\route_kaohsiung_sihanoukville.csv


## 8. Folium Map & Open in Browser

In [22]:
m = folium.Map(location=[(START_LAT+END_LAT)/2, (START_LON+END_LON)/2], zoom_start=5, tiles="cartodbpositron")

# Add land polygons (simplified for web map)
try:
    land_simplified = land.simplify(0.01, preserve_topology=True)
    folium.GeoJson(land_simplified.__geo_interface__, name="Land", style_function=lambda x: {
        "fillColor": "#e6e6e6", "color": "#333333", "weight": 0.5, "fillOpacity": 1.0
    }).add_to(m)
except Exception:
    pass

# Add our buffered land (what feasibility actually uses)
try:
    folium.GeoJson(land_buf.__geo_interface__, name="Buffered Land (0.5 km)",
                   style_function=lambda x: {"fillColor": "#f2f2f2", "color": "#999999", "weight": 0.5, "fillOpacity": 1.0}
                  ).add_to(m)
except Exception:
    pass


# Add corridor if exists
if corridor_polygon is not None:
    try:
        folium.GeoJson(corridor_polygon.__geo_interface__, name="Corridor", style_function=lambda x: {
            "fillColor": "#93c5fd", "color": "#60a5fa", "weight": 0.5, "fillOpacity": 0.3
        }).add_to(m)
    except Exception:
        pass

# Add route
folium.PolyLine([(lat, lon) for (lon,lat) in best_route], color="#1f2937", weight=4, opacity=0.9, tooltip=f"Best route: {best_dist:.2f} km").add_to(m)

# Markers
folium.Marker(location=[START_LAT, START_LON], popup="Start: Kaohsiung").add_to(m)
folium.Marker(location=[END_LAT, END_LON], popup="End: Sihanoukville").add_to(m)

html_path = os.path.join(OUTPUT_DIR, HTML_NAME)
m.save(html_path)
print("Saved map:", html_path)

# Auto-open (works on local VSCode environment)
abs_html = os.path.abspath(html_path)
try:
    webbrowser.open(f"file:///{abs_html}")
except Exception as e:
    print("Open in browser failed, please open manually:", abs_html)

Saved map: outputs\route_kaohsiung_sihanoukville.html


## Debug

In [23]:
print("AOI =", AOI)
print("land.empty =", land.empty)
print("land_buf.empty =", land_buf.empty)

print("land_polys (buffer後 explode) =", len(land_polys))
# 隨機挑一個多邊形看 bbox
if len(land_polys) > 0:
    bb = land_polys[0].bounds
    print("sample land_poly[0] bounds:", bb)


AOI = (100.52958, 7.60932, 123.27, 25.62)
land.empty = False
land_buf.empty = False
land_polys (buffer後 explode) = 295
sample land_poly[0] bounds: (119.75121988829164, 13.169645140465212, 123.27449157642059, 18.650195567536052)


In [24]:
# 先確保 densify_geodesic / path_crosses_land 已定義
print("直連 start→end 是否穿陸？", path_crosses_land([start, end]))


直連 start→end 是否穿陸？ True


In [25]:
# 把 start→end 這段 densify 成折線，看看能 query 出多少陸地候選
pts_test = densify_geodesic(start, end, max_step_km=10.0)
line_test = LineString(pts_test)
cands = land_tree.query(line_test)
print("候選多邊形數 =", len(cands))
# 看看前幾個候選的 bounds
for i, c in enumerate(list(cands)[:5]):
    poly_i = _as_poly(c)
    print(f"cand[{i}] type={type(poly_i)} bounds={getattr(poly_i,'bounds',None)}")


候選多邊形數 = 141
cand[0] type=<class 'shapely.geometry.polygon.Polygon'> bounds=(117.18375147493722, 8.330340819011889, 119.72983852326419, 11.427727160434353)
cand[1] type=<class 'shapely.geometry.polygon.Polygon'> bounds=(120.0723430399969, 11.359230623420672, 120.11542250438724, 11.395011599430562)
cand[2] type=<class 'shapely.geometry.polygon.Polygon'> bounds=(120.16267450226937, 11.429667771340844, 120.20111339018139, 11.468214372078517)
cand[3] type=<class 'shapely.geometry.polygon.Polygon'> bounds=(120.1291460800804, 11.739647768644929, 120.20095377158805, 11.787828682851885)
cand[4] type=<class 'shapely.geometry.polygon.Polygon'> bounds=(119.74812828342381, 10.441699075079006, 120.01572949027171, 10.64917137811915)


# EvoSwarm + 陸地 + Scgraph海節點/線

In [1]:
# ============================================================
# Evo+Swarm (GA+PSO) Sea Route Planner — Kaohsiung → Sihanoukville
# Single-cell version (paste and run)
# ============================================================

# =========================
# USER CONFIGURATION
# =========================
LAND_SHP = r"C:\Users\slab\Desktop\Slab Project\Stage1\data\Land\ne_10m_land.shp"

# Start & End (Kaohsiung, Sihanoukville)
START_LON, START_LAT = 120.27, 22.62
END_LON,   END_LAT   = 103.52958, 10.60932

# AOI bbox padding (deg)
AOI_PADDING_DEG = 3.0

# Corridor parameters
USE_SCGRAPH_CORRIDOR   = True     # 若沒安裝 scgraph，改 False
CORRIDOR_HALF_WIDTH_M  = 3000.0   # 走廊半寬 3 km
CORRIDOR_PENALTY       = 0.02     # fitness 的微小引導權重（占比小）

# Land safety buffer (meters) — 這裡用 1 km 比 0.5 km 更穩定
LAND_BUFFER_M = 1000.0

# Waypoints (excluding start/end)
K_WAYPOINTS = 8

# Evo+Swarm parameters — 先用小參數快速觀察
RANDOM_SEED = 42
GA_POP = 16
GA_GENS = 20
GA_CROSSOVER_RATE = 0.8
GA_MUTATION_RATE  = 0.25
GA_ELITISM = 2

PSO_PARTICLES = 8
PSO_ITERS     = 8
PSO_W_START = 0.9
PSO_W_END   = 0.4
PSO_C1 = 1.6
PSO_C2 = 1.6

# Feature guidance（軟引導）
LAMBDA_CORRIDOR = 0.02   # 越靠近中心線越好（影響很小）
LAMBDA_FEATURE  = 0.01   # 越靠近尖角點越好（影響很小）

# Outputs
OUTPUT_DIR   = "outputs"
HTML_NAME    = "route_kaohsiung_sihanoukville.html"
GEOJSON_NAME = "route_kaohsiung_sihanoukville.geojson"
CSV_NAME     = "route_kaohsiung_sihanoukville.csv"

# ============================================================
# Imports
# ============================================================
import os, math, random, json, warnings, webbrowser
from typing import List, Tuple, Optional
import numpy as np
import geopandas as gpd

from shapely.geometry import LineString, MultiLineString, Point, Polygon, box
from shapely.ops import nearest_points
from shapely import STRtree

from pyproj import CRS, Transformer, Geod

warnings.filterwarnings("ignore")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 我們自己的模組
from routing import features as RFEAT
from routing import scgraph_bridge as RSGB
from routing.geodesy import (
    to_m, to_ll,
    geodesic_km,  # 需要你在 geodesy.py 內加 alias: geodesic_km = gc_distance_km
    geodesic_azimuth,
    geodesic_midpoint,
    densify_geodesic,  # 需要你把它也放在 geodesy.py 內
    geodesic_fwd       # 需要新增到 geodesy.py 內（前進解）
)

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
GEOD = Geod(ellps="WGS84")

# ============================================================
# Utilities
# ============================================================
def compute_aoi(start, end, pad=2.0):
    x0 = min(start[0], end[0]) - pad
    y0 = min(start[1], end[1]) - pad
    x1 = max(start[0], end[0]) + pad
    y1 = max(start[1], end[1]) + pad
    return (x0, y0, x1, y1)

def polyline_length_km(coords: List[Tuple[float,float]]) -> float:
    return sum(geodesic_km(coords[i], coords[i+1]) for i in range(len(coords)-1))

def ensure_crs4326(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    if gdf.crs is None:
        gdf = gdf.set_crs(4326)
    else:
        gdf = gdf.to_crs(4326)
    return gdf

def project_local_m(gdf: gpd.GeoDataFrame) -> Tuple[gpd.GeoDataFrame, CRS, CRS, Transformer, Transformer]:
    crs_wgs = CRS.from_epsg(4326)
    crs_m   = CRS.from_epsg(3857)
    to_mtr  = Transformer.from_crs(crs_wgs, crs_m, always_xy=True)
    to_wgs  = Transformer.from_crs(crs_m, crs_wgs, always_xy=True)
    return gdf.to_crs(crs_m), crs_wgs, crs_m, to_mtr, to_wgs

# ============================================================
# 2. Load land & build STRtree
# ============================================================
start = (START_LON, START_LAT)
end   = (END_LON,   END_LAT)
AOI   = compute_aoi(start, end, AOI_PADDING_DEG)

land = gpd.read_file(LAND_SHP)
land = ensure_crs4326(land)

# Clip to AOI
aoi_poly = gpd.GeoDataFrame(geometry=[box(*AOI)], crs=4326)
try:
    land = land.clip(aoi_poly)
except Exception:
    land = gpd.overlay(land, aoi_poly, how="intersection")

# Buffer land（安全 1 km）
land_m, _, _, _, _ = project_local_m(land)
land_buf_m = land_m.buffer(LAND_BUFFER_M)
land_buf = gpd.GeoDataFrame(geometry=land_buf_m, crs=land_m.crs).to_crs(4326)

# Explode to polygons & STRtree
land_exploded = land_buf.explode(index_parts=False, ignore_index=True)
land_exploded = land_exploded[land_exploded.geometry.notnull() & (~land_exploded.geometry.is_empty)]
from shapely.geometry import Polygon as _Poly, MultiPolygon as _MPoly
land_polys: List[Polygon] = []
for geom in land_exploded.geometry:
    if isinstance(geom, (_Poly, _MPoly)):
        land_polys.append(geom)
land_tree = STRtree(land_polys)

# ============================================================
# 3. Corridor from scgraph (optional)
# ============================================================
corridor_centerline: Optional[MultiLineString] = None
corridor_polygon:    Optional[Polygon] = None

def segments_to_multiline(segs):
    ls = []
    for (a, b) in segs or []:
        try:
            ls.append(LineString([a, b]))
        except Exception:
            continue
    if not ls:
        return None
    return MultiLineString(ls)

if USE_SCGRAPH_CORRIDOR:
    try:
        got = RSGB.sc_edges_in_bbox(
            bbox_ll=AOI, edge_sample_ratio=0.8, max_sample_routes=40
        )
        edges = got.get("edges", [])
        ml = segments_to_multiline(edges)
        if ml is not None:
            gdf_ml = gpd.GeoDataFrame(geometry=[ml], crs=4326).to_crs(3857)
            corridor_polygon_m = gdf_ml.buffer(CORRIDOR_HALF_WIDTH_M)
            corridor_gdf = gpd.GeoDataFrame(geometry=corridor_polygon_m.geometry, crs=3857).to_crs(4326)
            corridor_gdf = corridor_gdf.explode(index_parts=False, ignore_index=True)
            corridor_gdf = corridor_gdf[corridor_gdf.geometry.notnull() & (~corridor_gdf.geometry.is_empty)]
            corridor_polygon = corridor_gdf.unary_union
            corridor_centerline = ml
            print("Corridor constructed from scgraph lines.")
        else:
            print("No maritime lines found for corridor; proceed without corridor.")
    except Exception as e:
        print("Corridor generation failed; proceed without corridor. Error:", e)

# ============================================================
# 4. Feature points (coast offset curvature peaks)
# ============================================================
try:
    feat = RFEAT.extract_feature_points_bbox(
        shp_path=LAND_SHP,
        bbox_ll_polygon=box(*AOI),
        avoid_km=max( (LAND_BUFFER_M/1000.0)+2.0, 7.0 ),
        simplify_m=1000.0,
        ANGLE_CONVEX_MAX=170.0,
        ANGLE_CONCAVE_MIN=210.0,
        MIN_PROM_CONVEX=0.002,
        MIN_PROM_CONCAVE=0.002,
        DEDUP_CONVEX_M=800.0,
        DEDUP_CONCAVE_M=800.0,
        ENABLE_UNIFORM=False
    )
    feature_points = list(set( tuple(map(float,p)) for p in (feat.get("convex",[])+feat.get("convex_peaks",[])) ))
    print(f"Feature points loaded: {len(feature_points)}")
except Exception as e:
    print("Feature point extraction failed; continue without features. Error:", e)
    feature_points = []

# 最近特徵點距離（若 scipy 可用就 KDTree）
_kdtree = None
try:
    from scipy.spatial import cKDTree
    if feature_points:
        _kdtree = cKDTree(np.array(feature_points, dtype=float))
except Exception:
    _kdtree = None

def nearest_feature_dist_km(pt: Tuple[float,float]) -> float:
    if not feature_points:
        return 0.0
    if _kdtree is not None:
        d, idx = _kdtree.query(np.array(pt, dtype=float), k=1)
        tgt = feature_points[int(idx)]
        return geodesic_km(pt, (tgt[0], tgt[1]))
    # fallback: 線性掃描
    best = 1e18
    for q in feature_points:
        best = min(best, geodesic_km(pt, q))
    return best if best<1e18 else 0.0

# ============================================================
# 5. Land crossing test & snapping
# ============================================================
def _as_poly(obj):
    if isinstance(obj, (int, np.integer)) and 0 <= int(obj) < len(land_polys):
        return land_polys[int(obj)]
    return obj

def segment_crosses_land(p0, p1) -> bool:
    pts = densify_geodesic(p0, p1, max_step_km=5.0)
    line = LineString(pts)
    candidates = land_tree.query(line)
    for c in candidates:
        poly = _as_poly(c)
        if isinstance(poly, (_Poly, _MPoly)) and line.intersects(poly):
            return True
    return False

def path_crosses_land(coords: List[Tuple[float,float]]) -> bool:
    for i in range(len(coords)-1):
        if segment_crosses_land(coords[i], coords[i+1]):
            return True
    return False

def project_point_to_sea(pt):
    """若在緩衝陸地內，推到最近海側邊界。"""
    p = Point(pt)
    candidates = land_tree.query(p)
    hit = None
    for c in candidates:
        poly = _as_poly(c)
        if isinstance(poly, (_Poly, _MPoly)) and p.within(poly):
            hit = poly
            break
    if hit is None:
        return pt
    boundary = hit.boundary if isinstance(hit, _MPoly) else hit.exterior
    np1, np2 = nearest_points(p, boundary)
    return (float(np2.x), float(np2.y))

def project_point_to_corridor(pt):
    if corridor_polygon is None:
        return pt
    p = Point(pt)
    if corridor_polygon.contains(p):
        return pt
    np1, np2 = nearest_points(p, corridor_polygon)
    return (float(np2.x), float(np2.y))

# ============================================================
# 6. Feature-guided segment repair (拓撲繞行器)
# ============================================================
def _normal_azimuth_deg(a: Tuple[float,float], b: Tuple[float,float]) -> float:
    """回傳 a->b 的地理方位 +90 度（外推法向）"""
    az = geodesic_azimuth(a, b)
    return (az + 90.0) % 360.0

def _insert_point(lst, idx, pt):
    return lst[:idx+1] + [pt] + lst[idx+1:]

def _snap_full(pt):
    pt2 = project_point_to_sea(pt)
    if corridor_centerline is not None:
        pt2 = project_point_to_corridor(pt2)
    return pt2

def _nearest_centerline_point(pt) -> Optional[Tuple[float,float]]:
    if corridor_centerline is None:
        return None
    p = Point(pt)
    try:
        np1, np2 = nearest_points(p, corridor_centerline)
        return (float(np2.x), float(np2.y))
    except Exception:
        return None

def _nearest_feature_point(pt) -> Optional[Tuple[float,float]]:
    if not feature_points:
        return None
    if _kdtree is not None:
        d, idx = _kdtree.query(np.array(pt, dtype=float), k=1)
        q = feature_points[int(idx)]
        return (float(q[0]), float(q[1]))
    # fallback
    best, tgt = 1e18, None
    for q in feature_points:
        dist = geodesic_km(pt, q)
        if dist < best:
            best, tgt = dist, q
    return tgt

def repair_route_to_sea(coords: List[Tuple[float,float]], max_inserts: int = 8) -> Tuple[List[Tuple[float,float]], bool, str]:
    """
    當某一段穿陸：優先把插點吸到『走廊中心』或『尖角特徵點』，仍不行再用法向外推
    回傳：(新座標, 是否可行, 說明)
    """
    route = coords[:]
    note  = ""
    inserts = 0

    # 先把所有點 snap 一次（處理起迄點貼岸）
    route = [_snap_full(p) for p in route]

    while inserts < max_inserts and path_crosses_land(route):
        # 找第一條穿陸段
        bad_i = None
        for i in range(len(route)-1):
            if segment_crosses_land(route[i], route[i+1]):
                bad_i = i
                break
        if bad_i is None:
            break

        a, b = route[bad_i], route[bad_i+1]
        # 以 geodesic densify 取中點
        mid = geodesic_midpoint(a, b)

        # 優先：走廊中心最近點
        p_try = _nearest_centerline_point(mid)
        strategy = "centerline"
        if p_try is None:
            # 次之：尖角特徵點
            p_try = _nearest_feature_point(mid)
            strategy = "feature"

        # 實在沒有 → 法向外推階梯
        if p_try is None:
            strategy = "normal-push"
            az = _normal_azimuth_deg(a, b)
            for delta in (20, 40, 80, 120, 160):  # km
                cand = geodesic_fwd(mid[0], mid[1], az, delta)
                cand = _snap_full(cand)
                if not segment_crosses_land(a, cand) and not segment_crosses_land(cand, b):
                    p_try = cand
                    break
            if p_try is None:
                # 反向再試
                az2 = (az + 180.0) % 360.0
                for delta in (20, 40, 80, 120, 160):
                    cand = geodesic_fwd(mid[0], mid[1], az2, delta)
                    cand = _snap_full(cand)
                    if not segment_crosses_land(a, cand) and not segment_crosses_land(cand, b):
                        p_try = cand
                        break

        if p_try is None:
            # 仍然不行：小幅隨機抖動
            p_try = _snap_full( (mid[0] + np.random.normal(0, 0.1), mid[1] + np.random.normal(0, 0.1)) )
            strategy = "random"

        # 插點並記錄
        route = _insert_point(route, bad_i, _snap_full(p_try))
        inserts += 1
        note = f"inserted={inserts}, last_strategy={strategy}"

    feasible = not path_crosses_land(route)
    return route, feasible, note

# ============================================================
# 7. Initialization (混合取樣 + 段級修復)
# ============================================================
def linspace_route(a, b, k):
    xs = np.linspace(a[0], b[0], k+2)[1:-1]
    ys = np.linspace(a[1], b[1], k+2)[1:-1]
    return list(zip(xs, ys))

def init_individual(k: int) -> List[Tuple[float,float]]:
    mids = linspace_route(start, end, k)
    # 混合取樣策略：大多數先靠走廊，小部分靠尖角點
    for i in range(len(mids)):
        r = np.random.rand()
        if corridor_centerline is not None and r < 0.8:
            # 在 AOI 均勻取樣後投影到走廊
            rx = np.random.uniform(AOI[0], AOI[2])
            ry = np.random.uniform(AOI[1], AOI[3])
            mids[i] = project_point_to_corridor((rx, ry))
        elif feature_points and r < 0.95:
            # 尖角特徵附近高斯
            q = feature_points[np.random.randint(0, len(feature_points))]
            dx = np.random.normal(0, 0.15)  # deg 級距，約 10–20 km
            dy = np.random.normal(0, 0.15)
            mids[i] = (q[0]+dx, q[1]+dy)
        else:
            # 純隨機 + snap
            rx = np.random.uniform(AOI[0], AOI[2])
            ry = np.random.uniform(AOI[1], AOI[3])
            mids[i] = (rx, ry)

        mids[i] = _snap_full(mids[i])

    route = [ _snap_full(start) ] + mids + [ _snap_full(end) ]
    # 段級修復，保證種子盡量可行
    route, feasible, note = repair_route_to_sea(route, max_inserts=8)
    if not feasible:
        # 最後保底：S→T 中點外推一個樞紐
        mp = geodesic_midpoint(start, end)
        az = _normal_azimuth_deg(start, end)
        hub = _snap_full( geodesic_fwd(mp[0], mp[1], az, 120.0) )
        route = [ _snap_full(start), hub, _snap_full(end) ]
        route, feasible, note = repair_route_to_sea(route, max_inserts=8)
    return route

# ============================================================
# 8. Fitness（硬約束：不可行=淘汰；主目標=距離；微引導=走廊/特徵）
# ============================================================
def corridor_offset_metric(coords: List[Tuple[float,float]]) -> float:
    if corridor_centerline is None:
        return 0.0
    acc, n = 0.0, 0
    for c in coords:
        try:
            np1, np2 = nearest_points(Point(c), corridor_centerline)
            acc += geodesic_km((np1.x, np1.y), (np2.x, np2.y))
            n += 1
        except Exception:
            pass
    if n == 0:
        return 0.0
    length_km = max(1.0, polyline_length_km(coords))
    return (acc / n) / length_km  # normalize

def feature_offset_metric(coords: List[Tuple[float,float]]) -> float:
    if not feature_points:
        return 0.0
    acc, n = 0.0, 0
    for c in coords:
        acc += nearest_feature_dist_km(c)
        n += 1
    if n == 0:
        return 0.0
    length_km = max(1.0, polyline_length_km(coords))
    return (acc / n) / length_km

def fitness(coords):
    # 硬限制：只要穿陸，直接當作不可行
    if path_crosses_land(coords):
        return -1e12  # 或用 -np.inf，但有些地方用不了 inf 就用超大負數
    L = polyline_length_km(coords)
    Dcorr = corridor_offset_metric(coords) if corridor_centerline is not None else 0.0
    return -L - (CORRIDOR_PENALTY * Dcorr)


# ============================================================
# 9. PSO local refine（只動內點，並做 snap/修復）
# ============================================================
def pso_refine(coords: List[Tuple[float,float]], iters=PSO_ITERS, particles=PSO_PARTICLES):
    interior = coords[1:-1]
    if len(interior) == 0:
        return coords[:]
    x0 = np.array([v for pt in interior for v in pt], dtype=float)

    x_min = []; x_max = []
    for _ in range(len(x0)//2):
        x_min += [AOI[0], AOI[1]]
        x_max += [AOI[2], AOI[3]]
    x_min = np.array(x_min); x_max = np.array(x_max)

    def vec_to_coords(vec):
        mids = [(vec[2*i], vec[2*i+1]) for i in range(len(vec)//2)]
        return [coords[0]] + mids + [coords[-1]]

    def clamp_vec(vec):
        return np.clip(vec, x_min, x_max)

    rng = np.random.default_rng(RANDOM_SEED)
    swarm_x = np.tile(x0, (particles,1)) + rng.normal(0, 0.01*(x_max-x_min), size=(particles, len(x0)))
    swarm_v = rng.normal(0, 0.005*(x_max-x_min), size=(particles, len(x0)))

    pbest_x = swarm_x.copy()
    pbest_f = np.array([fitness(vec_to_coords(xx)) for xx in pbest_x])

    idxs = np.arange(particles)
    left = (idxs - 1) % particles
    right = (idxs + 1) % particles

    def inertia_weight(t):
        return PSO_W_START + (PSO_W_END - PSO_W_START) * (t / max(1, iters-1))

    for t in range(iters):
        lbest = []
        for i in range(particles):
            neighbors = [i, left[i], right[i]]
            j = neighbors[np.argmax(pbest_f[neighbors])]
            lbest.append(pbest_x[j])
        lbest = np.array(lbest)

        w = inertia_weight(t)
        r1 = rng.random(size=swarm_x.shape)
        r2 = rng.random(size=swarm_x.shape)

        swarm_v = (w*swarm_v + PSO_C1*r1*(pbest_x - swarm_x) + PSO_C2*r2*(lbest - swarm_x))
        swarm_x = clamp_vec(swarm_x + swarm_v)

        # snap/修復
        for i in range(particles):
            mids = [(swarm_x[i,2*j], swarm_x[i,2*j+1]) for j in range(len(x0)//2)]
            mids2 = [ _snap_full(pt) for pt in mids ]
            # 回寫
            for j,pt in enumerate(mids2):
                swarm_x[i,2*j]   = pt[0]
                swarm_x[i,2*j+1] = pt[1]

        fvals = np.array([fitness(vec_to_coords(xx)) for xx in swarm_x])
        improved = fvals > pbest_f
        pbest_x[improved] = swarm_x[improved]
        pbest_f[improved] = fvals[improved]

    j = int(np.argmax(pbest_f))
    best = vec_to_coords(pbest_x[j])
    # 最後再跑一次段級修復，確保可行
    best, feasible, note = repair_route_to_sea(best, max_inserts=6)
    return best

# ============================================================
# 10. GA Outer Loop
# ============================================================
def tournament_select(pop, fits, k=3):
    idxs = np.random.choice(len(pop), size=k, replace=False)
    best = idxs[0]
    for i in idxs[1:]:
        if fits[i] > fits[best]:
            best = i
    return pop[best]

def crossover(parent1, parent2):
    p1_mid = parent1[1:-1]; p2_mid = parent2[1:-1]
    if len(p1_mid) == 0:
        return parent1[:]
    r = np.random.rand()
    if r < 0.5:
        # splice
        cut = np.random.randint(1, len(p1_mid))
        child_mid = p1_mid[:cut] + p2_mid[cut:]
    else:
        # BLX-like blend
        child_mid = []
        for (a,b) in zip(p1_mid, p2_mid):
            alpha = np.random.uniform(-0.2, 1.2)
            cx = a[0] + alpha*(b[0]-a[0])
            cy = a[1] + alpha*(b[1]-a[1])
            child_mid.append((cx, cy))
    child_mid = [ _snap_full(pt) for pt in child_mid ]
    return [parent1[0]] + child_mid + [parent1[-1]]

def mutate(ind, rate=GA_MUTATION_RATE):
    mids = ind[1:-1]
    if not mids:
        return ind[:]
    x0,y0,x1,y1 = AOI
    lon_scale = (x1-x0) * 0.01
    lat_scale = (y1-y0) * 0.01
    mids2 = []
    for pt in mids:
        if np.random.rand() < rate:
            pt = (pt[0] + np.random.normal(0, lon_scale),
                  pt[1] + np.random.normal(0, lat_scale))
        mids2.append(_snap_full(pt))
    return [ind[0]] + mids2 + [ind[-1]]

def evo_swarm_optimize():
    pop = init_population_with_seeds(
        pop_size=GA_POP,
        k_interior=K_WAYPOINTS,
        ratios=(0.4, 0.4)   # 40% scgraph + 40% 海岸特徵 + 20% 原本隨機
    )

    fits = np.array([fitness(ind) for ind in pop])

    best_route = pop[int(np.argmax(fits))]
    best_fit   = float(np.max(fits))
    hist = [(-best_fit, best_fit)]

    for g in range(GA_GENS):
        feasible_mask = np.array([not path_crosses_land(ind) for ind in pop])
        print(f"Gen {g+1:03d} | feasible: {feasible_mask.mean():.0%}")
        new_pop = []

        # elitism
        elite_idx = np.argsort(-fits)[:GA_ELITISM]
        for i in elite_idx:
            new_pop.append(pop[i])

        while len(new_pop) < GA_POP:
            p1 = tournament_select(pop, fits, k=3)
            p2 = tournament_select(pop, fits, k=3)
            child = p1
            if np.random.rand() < GA_CROSSOVER_RATE:
                child = crossover(p1, p2)
            child = mutate(child, rate=GA_MUTATION_RATE)
            # local refine + 修復
            child = pso_refine(child, iters=PSO_ITERS, particles=PSO_PARTICLES)
            new_pop.append(child)

        pop  = new_pop
        fits = np.array([fitness(ind) for ind in pop])

        idx = int(np.argmax(fits))
        if fits[idx] > best_fit:
            best_fit   = float(fits[idx])
            best_route = pop[idx]

        if (g+1) % 5 == 0 or g == 0:
            dist_km = polyline_length_km(best_route)
            print(f"Gen {g+1:3d} | best distance: {dist_km:.2f} km")
            hist.append((dist_km, best_fit))

    return best_route, hist

# ====== 1) 小工具：將 polyline 縮成 K 個內點（保持端點） ======
from shapely.geometry import LineString as _LS2

def _evenly_resample_polyline(coords, k_interior):
    """把一條 polyline 均勻取樣成: start + k_interior 個中繼點 + end"""
    if len(coords) < 2:
        return coords
    ls = _LS2(coords)
    L = ls.length
    if L <= 0 or k_interior <= 0:
        return [coords[0], coords[-1]]
    ts = [i/(k_interior+1) for i in range(0, k_interior+2)]
    pts = [ls.interpolate(t*L) for t in ts]
    out = [(float(p.x), float(p.y)) for p in pts]
    # shapely 的 interpolate 在經緯度視作平面，這裡只是取比例點，OK
    return out

def _shapely_ls_from_lonlat(coords):
    return _LS2([(lon, lat) for (lon,lat) in coords])

def _project_waypoints_to_sea_and_corridor(mids):
    mids2 = []
    for pt in mids:
        pt = project_point_to_sea(pt)
        if corridor_centerline is not None:
            pt = project_point_to_corridor(pt)
        mids2.append(pt)
    return mids2

def _guarantee_start_end_sea(s, e):
    return project_point_to_sea(s), project_point_to_sea(e)

# ====== 2) 種子 A：scgraph 路徑簡化（保底可行路線） ======
def build_seed_from_scgraph(start, end, k_interior):
    try:
        # 你提供的橋接層
        from routing.scgraph_bridge import sc_shortest_path_lonlat
    except Exception:
        return None
    out = sc_shortest_path_lonlat(start, end)
    track = out.get("track") if isinstance(out, dict) else None
    if not track or len(track) < 2:
        return None
    # 均勻抽成 K+2 個點
    coarse = _evenly_resample_polyline(track, k_interior)
    # 保底：端點用我們的 start/end（避免 sc 的端點不在海上）
    coarse[0]  = start
    coarse[-1] = end
    mids = coarse[1:-1]
    mids = _project_waypoints_to_sea_and_corridor(mids)
    seed = [start] + mids + [end]
    # 若仍穿陸就放棄（後續還有其它種子）
    if path_crosses_land(seed):
        return None
    return seed

# ====== 3) 種子 B：海岸特徵點骨架（convex_peaks） ======
def _straight_line():
    return _LS2([start, end])

def build_seed_from_coast_features(k_interior, max_peaks=3, search_km=200.0):
    """
    從海岸外推的凸峰（尖角）裡挑 1~max_peaks 個，沿著 start→end 的順序串成骨架。
    挑選準則：距離「起訖直線」近、且投影在線段之間（避免太遠/回頭繞）。
    """
    try:
        from routing.features import extract_feature_points_bbox
    except Exception:
        return None
    # 準備 AOI polygon（略放大，確保抓到鄰近特徵）
    x0,y0,x1,y1 = AOI
    pad = 0.5  # 度，給特徵點一些緩衝
    bbox = box(x0-pad, y0-pad, x1+pad, y1+pad)

    # 抽特徵點（用你現成的函式）
    feats = extract_feature_points_bbox(
        shp_path=LAND_SHP,
        bbox_ll_polygon=bbox,
        avoid_km=15.0,  # 與你 features.py 預設一致
    )
    peaks = feats.get("convex_peaks", []) or feats.get("convex", [])
    if not peaks:
        return None

    # 過濾「在起訖線段投影之外」的點（避免回頭）
    base = _straight_line()
    cand = []
    for (lon,lat) in peaks:
        p = Point(lon,lat)
        proj = base.project(p)  # [0, length]
        if 1e-6 < proj < base.length - 1e-6:
            # 距離直線的平面距離（度量近似即可；只是排序用）
            d = base.distance(p)
            cand.append(((lon,lat), d, proj))
    if not cand:
        return None

    # 取距離直線最近的若干個點，並按投影位置從 start→end 排序
    cand.sort(key=lambda t: (t[1], t[2]))
    chosen = [c[0] for c in cand[:max_peaks]]

    # 若選太多，均勻抽到 k_interior
    if len(chosen) > k_interior:
        idx = np.linspace(0, len(chosen)-1, num=k_interior, dtype=int)
        chosen = [chosen[i] for i in idx]
    # 若選太少，後續 PSO/GA 還有自由度，不強填

    mids = _project_waypoints_to_sea_and_corridor(chosen)
    seed = [start] + mids + [end]
    if path_crosses_land(seed):
        return None
    return seed

# ====== 4) 混合初始化：把兩類種子混進第一代 ======
def init_population_with_seeds(pop_size, k_interior, ratios=(0.4, 0.4)):
    """
    ratios: (scgraph_seed_ratio, coast_feature_seed_ratio)
    剩下的用原本的 init_individual() 填滿。
    """
    sc_ratio, cf_ratio = ratios
    n_sc = int(round(pop_size * sc_ratio))
    n_cf = int(round(pop_size * cf_ratio))
    pop = []

    # 先確保起訖在海上
    s_fixed, e_fixed = _guarantee_start_end_sea(start, end)
    # 用局部變數覆蓋一下（不改外部全域，避免其他地方混亂）
    _s, _e = s_fixed, e_fixed

    # 1) scgraph 種子
    for _ in range(n_sc):
        seed = build_seed_from_scgraph(_s, _e, k_interior)
        if seed is not None:
            pop.append(seed)

    # 2) 海岸特徵點種子
    for _ in range(n_cf):
        seed = build_seed_from_coast_features(k_interior, max_peaks=min(3, k_interior))
        if seed is not None:
            pop.append(seed)

    # 3) 用舊的隨機擾動種子補滿
    while len(pop) < pop_size:
        ind = init_individual(k_interior)
        pop.append(ind)

    return pop


# ============================================================
# 11. Run
# ============================================================
best_route, hist = evo_swarm_optimize()
best_dist = polyline_length_km(best_route)
print(f"\nBest distance = {best_dist:.2f} km")

# ============================================================
# 12. Save outputs
# ============================================================
geo = {
    "type": "FeatureCollection",
    "features": [{
        "type": "Feature",
        "properties": {"name": "Kaohsiung→Sihanoukville", "distance_km": best_dist},
        "geometry": {"type": "LineString", "coordinates": [[lon,lat] for (lon,lat) in best_route]}
    }]
}
geo_path = os.path.join(OUTPUT_DIR, GEOJSON_NAME)
with open(geo_path, "w", encoding="utf-8") as f:
    json.dump(geo, f, ensure_ascii=False, indent=2)

csv_path = os.path.join(OUTPUT_DIR, CSV_NAME)
with open(csv_path, "w", encoding="utf-8") as f:
    f.write("lon,lat\n")
    for lon,lat in best_route:
        f.write(f"{lon},{lat}\n")

print("Saved:", geo_path)
print("Saved:", csv_path)

# ============================================================
# 13. Folium map
# ============================================================
import folium

m = folium.Map(location=[(START_LAT+END_LAT)/2, (START_LON+END_LON)/2], zoom_start=5, tiles="cartodbpositron")

# Land
try:
    land_simplified = land.simplify(0.01, preserve_topology=True)
    folium.GeoJson(land_simplified.__geo_interface__, name="Land",
                   style_function=lambda x: {"fillColor":"#e6e6e6","color":"#333","weight":0.5,"fillOpacity":1.0}).add_to(m)
except Exception:
    pass

# Buffered Land (safety)
try:
    folium.GeoJson(land_buf.__geo_interface__, name="Buffered Land (1 km)",
                   style_function=lambda x: {"fillColor":"#f2f2f2","color":"#999","weight":0.5,"fillOpacity":1.0}).add_to(m)
except Exception:
    pass

# Corridor
if corridor_polygon is not None:
    try:
        folium.GeoJson(corridor_polygon.__geo_interface__, name="Corridor",
                       style_function=lambda x: {"fillColor":"#93c5fd","color":"#60a5fa","weight":0.5,"fillOpacity":0.3}).add_to(m)
    except Exception:
        pass
    # centerline
    try:
        folium.GeoJson(corridor_centerline.__geo_interface__, name="Corridor centerline",
                       style_function=lambda x: {"color":"#2563eb","weight":1.5,"opacity":0.9}).add_to(m)
    except Exception:
        pass

# Feature points
if feature_points:
    for q in feature_points:
        folium.CircleMarker(location=[q[1], q[0]], radius=3,
                            color="#16a34a", fill=True, fill_opacity=0.8,
                            tooltip="feature").add_to(m)

# Straight line baseline
folium.PolyLine([(START_LAT, START_LON), (END_LAT, END_LON)],
                color="#ef4444", weight=2, opacity=0.6, tooltip="S→T straight").add_to(m)

# Best route
folium.PolyLine([(lat, lon) for (lon,lat) in best_route],
                color="#1f2937", weight=4, opacity=0.95,
                tooltip=f"Best route: {best_dist:.2f} km").add_to(m)

# Markers
folium.Marker(location=[START_LAT, START_LON], popup="Start: Kaohsiung").add_to(m)
folium.Marker(location=[END_LAT,   END_LON],   popup="End: Sihanoukville").add_to(m)

html_path = os.path.join(OUTPUT_DIR, HTML_NAME)
m.save(html_path)
print("Saved map:", html_path)

try:
    abs_html = os.path.abspath(html_path)
    webbrowser.open(f"file:///{abs_html}")
except Exception as e:
    print("Open in browser failed:", e)


Corridor constructed from scgraph lines.
Feature points loaded: 1613
Gen 001 | feasible: 0%
Gen   1 | best distance: 2198.82 km
Gen 002 | feasible: 0%
Gen 003 | feasible: 0%
Gen 004 | feasible: 0%
Gen 005 | feasible: 0%


KeyboardInterrupt: 

## Claude stage 1&2

In [ ]:
# ============================================================
# Sea Route Planner (scgraph-only / no corridor / robust repair / 1-opt simplify)
# 單一 Cell 可執行版
# ============================================================

# ---------------------
# User configuration
# ---------------------
LAND_SHP = r"C:\Users\slab\Desktop\Slab Project\Stage1\data\Land\ne_10m_land.shp"
START_LON, START_LAT = 120.27, 22.62         # Kaohsiung
END_LON,   END_LAT   = 103.52958, 10.60932   # Sihanoukville
AOI_PADDING_DEG = 6.0

# 推海：端點若落在陸地上，往外推的距離（km）
ENDPOINT_PUSH_OUT_KM = 5.0

# Land buffer（安全距離）：相當於「不能貼岸」的最小距離
LAND_BUFFER_M = 500.0

# 演化與局部最佳化
K_WAYPOINTS = 8
RANDOM_SEED = 42
GA_POP = 16
GA_GENS = 25
GA_CROSSOVER_RATE = 0.8
GA_MUTATION_RATE  = 0.25
GA_ELITISM = 2
PSO_PARTICLES = 8
PSO_ITERS     = 6
PSO_W_START, PSO_W_END = 0.9, 0.4
PSO_C1 = PSO_C2 = 1.6

# 幾何檢查精度（線段採樣步距，越小越嚴格；5~8km 建議值）
SEG_CHECK_STEP_KM = 6.0

# 輸出
OUTPUT_DIR   = "outputs"
HTML_NAME    = "route_kaohsiung_sihanoukville.html"
GEOJSON_NAME = "route_kaohsiung_sihanoukville.geojson"
CSV_NAME     = "route_kaohsiung_sihanoukville.csv"

# ============================================================
# Imports
# ============================================================
import os, math, json, random, warnings, webbrowser
from typing import List, Tuple, Optional
import numpy as np
import geopandas as gpd

from shapely.geometry import LineString, MultiLineString, Point, Polygon, box
from shapely.ops import nearest_points, unary_union
from shapely import STRtree
from shapely.prepared import prep

from pyproj import Geod, CRS, Transformer
warnings.filterwarnings("ignore")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# project modules
from routing import scgraph_bridge as RSGB
from routing.geodesy import (
    geodesic_km, geodesic_azimuth, geodesic_midpoint,
    geodesic_fwd, densify_geodesic
)

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
GEOD = Geod(ellps="WGS84")

# ============================================================
# Helpers
# ============================================================
def ensure_crs4326(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    if gdf.crs is None:
        return gdf.set_crs(4326)
    return gdf.to_crs(4326)

def compute_aoi(start, end, pad=2.0):
    x0 = min(start[0], end[0]) - pad
    y0 = min(start[1], end[1]) - pad
    x1 = max(start[0], end[0]) + pad
    y1 = max(start[1], end[1]) + pad
    return (x0, y0, x1, y1)

def polyline_length_km(coords: List[Tuple[float,float]]) -> float:
    return sum(geodesic_km(coords[i], coords[i+1]) for i in range(len(coords)-1))

# ============================================================
# Land load (buffered for避岸) + prepared union
# ============================================================
def _load_land_prepared(aoi):
    land = gpd.read_file(LAND_SHP)
    land = ensure_crs4326(land)
    aoi_poly = gpd.GeoDataFrame(geometry=[box(*aoi)], crs=4326)
    try:
        land = land.clip(aoi_poly)
    except Exception:
        land = gpd.overlay(land, aoi_poly, how="intersection")

    # buffer for safe sea mask
    land_m = land.to_crs(3857)
    land_buf = land_m.buffer(LAND_BUFFER_M)
    land_buf = gpd.GeoDataFrame(geometry=land_buf, crs=3857).to_crs(4326)
    land_exploded = land_buf.explode(index_parts=False, ignore_index=True)
    land_exploded = land_exploded[
        land_exploded.geometry.notnull() & (~land_exploded.geometry.is_empty)
    ]
    polys = [g for g in land_exploded.geometry if isinstance(g, (Polygon,))]
    union = unary_union(polys) if len(polys) else None
    return polys, prep(union) if union is not None else None

# Raw land（未 buffer）for 端點推海
def _load_land_raw(aoi):
    land = gpd.read_file(LAND_SHP)
    land = ensure_crs4326(land)
    aoi_poly = gpd.GeoDataFrame(geometry=[box(*aoi)], crs=4326)
    try:
        land = land.clip(aoi_poly)
    except Exception:
        land = gpd.overlay(land, aoi_poly, how="intersection")
    raw_polys = [g for g in land.explode(index_parts=False, ignore_index=True).geometry
                 if isinstance(g, (Polygon,))]
    return raw_polys, STRtree(raw_polys)

# ============================================================
# Endpoint snap & push-out
# ============================================================
def _point_within_any(poly_list, pt: Point) -> Optional[Polygon]:
    for poly in poly_list:
        if pt.within(poly):
            return poly
    return None

def project_endpoint_to_sea_unbuffered(pt, raw_polys) -> Tuple[float,float]:
    p = Point(pt)
    hit = _point_within_any(raw_polys, p)
    if hit is None:
        return pt
    bd = hit.exterior if hasattr(hit, "exterior") else hit.boundary
    _, q = nearest_points(p, bd)
    return (float(q.x), float(q.y))

def push_out_from_coast(pt, km: float, raw_polys) -> Tuple[float,float]:
    """沿法向由岸邊再推出去 km，以避免起訖點貼岸造成穿陸。"""
    p = Point(pt)
    hit = _point_within_any(raw_polys, p)
    if hit is None:
        # 若不在陸地上，仍往最近岸邊法向推出
        nearest = None; best = 1e18
        for poly in raw_polys:
            bd = poly.exterior if hasattr(poly,"exterior") else poly.boundary
            _, q = nearest_points(p, bd)
            d = geodesic_km((p.x,p.y),(q.x,q.y))
            if d < best:
                best, nearest = d, (float(q.x), float(q.y))
        if nearest is None:
            return pt
        # 由岸邊向外推出
        az = geodesic_azimuth(nearest, (pt[0], pt[1]))
        return geodesic_fwd(nearest[0], nearest[1], az, km)
    else:
        # 若在陸地，先投影到岸邊，再往外推
        bd = hit.exterior if hasattr(hit, "exterior") else hit.boundary
        _, q = nearest_points(p, bd)
        # 岸邊法向取向海側：以 q -> pt 的方向再延伸
        az = geodesic_azimuth((q.x,q.y), (pt[0],pt[1]))
        return geodesic_fwd(float(q.x), float(q.y), az, km)

# ============================================================
# Land tests
# ============================================================
land_polys: List[Polygon] = []
land_prepared = None

def segment_crosses_land(p0, p1) -> bool:
    if land_prepared is None:
        return False
    pts  = densify_geodesic(p0, p1, max_step_km=SEG_CHECK_STEP_KM)
    return land_prepared.intersects(LineString(pts))

def path_crosses_land(coords: List[Tuple[float,float]]) -> bool:
    return any(segment_crosses_land(coords[i], coords[i+1]) for i in range(len(coords)-1))

# ============================================================
# Robust repair: bisect + 360° fan search
# ============================================================
def _insert_point(lst, idx, pt):
    return lst[:idx+1] + [pt] + lst[idx+1:]

def _fan_try(a, b, origin, step_km=(10, 20, 40, 80, 120), az_step=30):
    for d in step_km:
        for az in range(0, 360, az_step):
            cand = geodesic_fwd(origin[0], origin[1], az, d)
            if not segment_crosses_land(a, cand) and not segment_crosses_land(cand, b):
                return cand
    return None

def repair_route_to_sea_bisect(route, max_inserts=96):
    r = route[:]
    inserts = 0
    while inserts < max_inserts and path_crosses_land(r):
        bad_i = None
        for i in range(len(r)-1):
            if segment_crosses_land(r[i], r[i+1]):
                bad_i = i
                break
        if bad_i is None:
            break

        a, b = r[bad_i], r[bad_i+1]
        mid = geodesic_midpoint(a, b)
        # 先把中點往海上移（若在陸上）
        cand0 = mid
        if any(Point(mid).within(poly) for poly in land_polys):
            # 投影到邊界
            for poly in land_polys:
                if Point(mid).within(poly):
                    bd = poly.exterior if hasattr(poly,"exterior") else poly.boundary
                    _, q = nearest_points(Point(mid), bd)
                    cand0 = (float(q.x), float(q.y))
                    break

        cand = _fan_try(a, b, cand0)
        if cand is None:
            # 再嘗試法向兩側
            az = (geodesic_azimuth(a, b) + 90.0) % 360.0
            for d in (20, 40, 80, 120, 160):
                for sign in (+1, -1):
                    p = geodesic_fwd(cand0[0], cand0[1], (az + 180*(sign<0)) % 360.0, d)
                    if not segment_crosses_land(a, p) and not segment_crosses_land(p, b):
                        cand = p; break
                if cand is not None: break
        if cand is None:
            # 最後隨機擾動
            cand = (cand0[0] + np.random.normal(0, 0.1),
                    cand0[1] + np.random.normal(0, 0.1))

        r = _insert_point(r, bad_i, cand)
        inserts += 1
    return r, (not path_crosses_land(r))

def repair_route_to_sea(coords: List[Tuple[float,float]], max_inserts: int = 96):
    r, ok = repair_route_to_sea_bisect(coords, max_inserts=max_inserts)
    return r, ok, ""

# ============================================================
# Seeds (scgraph only, with fallback straight-line)
# ============================================================
def _evenly_resample_polyline(coords, k_interior):
    if len(coords) < 2:
        return coords
    ls = LineString(coords)
    L = ls.length
    if L <= 0 or k_interior <= 0:
        return [coords[0], coords[-1]]
    ts = [i/(k_interior+1) for i in range(0, k_interior+2)]
    pts = [ls.interpolate(t*L) for t in ts]
    return [(float(p.x), float(p.y)) for p in pts]

def build_seed_from_scgraph(start, end, k_interior):
    track = None
    try:
        out = RSGB.sc_shortest_path_lonlat(start, end)
        track = out.get("track") if isinstance(out, dict) else None
    except Exception:
        track = None

    def _coarse_to_seed(track_coords):
        coarse = _evenly_resample_polyline(track_coords, k_interior+2)
        coarse[0] = start; coarse[-1] = end
        seed = [start] + coarse[1:-1] + [end]
        seed, ok, _ = repair_route_to_sea(seed, max_inserts=128)
        return seed, ok

    if track and len(track) >= 2:
        seed, ok = _coarse_to_seed(track)
        if ok:
            return seed

    # fallback：直線 → 修復
    seed = [start, end]
    seed, ok, _ = repair_route_to_sea(seed, max_inserts=160)
    if not ok:
        mp = geodesic_midpoint(start, end)
        helper = _fan_try(start, end, mp)
        if helper is not None:
            seed = [start, helper, end]
            seed, ok, _ = repair_route_to_sea(seed, max_inserts=160)
    if not ok:
        raise RuntimeError("仍無法建立可行種子；請放大 AOI 或檢查陸地底圖/緩衝。")
    return seed

def init_population_scgraph_only(pop_size, k_interior):
    base = build_seed_from_scgraph(start, end, k_interior)
    pop  = [base]  # baseline at index 0

    # 其餘個體：在可行基線附近做小擾動，再修復
    for _ in range(pop_size-1):
        mids = base[1:-1]
        if mids:
            vec = np.array(mids) + np.random.normal(0, 0.08, size=(len(mids),2))
            cand = [start] + [tuple(p) for p in vec] + [end]
        else:
            cand = base[:]
        cand, ok, _ = repair_route_to_sea(cand, max_inserts=64)
        pop.append(cand if ok else base[:])
    return pop

# ============================================================
# Fitness（重罰穿陸）
# ============================================================
def fitness(coords):
    L = polyline_length_km(coords)
    if not path_crosses_land(coords):
        return -L
    # 穿陸：每穿一次重罰 + 深度罰
    n_cross = 0; depth_pen = 0.0
    for i in range(len(coords)-1):
        if segment_crosses_land(coords[i], coords[i+1]):
            n_cross += 1
            mid = geodesic_midpoint(coords[i], coords[i+1])
            dmin = 1e9
            for poly in land_polys:
                if Point(mid).within(poly):
                    bd = poly.exterior if hasattr(poly,"exterior") else poly.boundary
                    _, q = nearest_points(Point(mid), bd)
                    dmin = min(dmin, geodesic_km((mid[0],mid[1]), (q.x,q.y)))
            if dmin < 1e9:
                depth_pen += dmin * 50.0
    return -(L + n_cross*500.0 + depth_pen)

# ============================================================
# PSO (小幅在 AOI 內微調)
# ============================================================
def pso_refine(coords, iters=PSO_ITERS, particles=PSO_PARTICLES):
    interior = coords[1:-1]
    if not interior:
        return coords[:]
    x0 = np.array([v for pt in interior for v in pt], dtype=float)
    x_min = np.tile([AOI[0], AOI[1]], len(interior))
    x_max = np.tile([AOI[2], AOI[3]], len(interior))

    def vec_to_coords(vec):
        mids = [(vec[2*i], vec[2*i+1]) for i in range(len(vec)//2)]
        return [coords[0]] + mids + [coords[-1]]

    def clamp_vec(vec):
        return np.clip(vec, x_min, x_max)

    rng = np.random.default_rng(RANDOM_SEED)
    swarm_x = np.tile(x0, (particles,1)) + rng.normal(0, 0.01*(x_max-x_min), size=(particles, len(x0)))
    swarm_v = rng.normal(0, 0.005*(x_max-x_min), size=(particles, len(x0)))
    pbest_x = swarm_x.copy()
    pbest_f = np.array([fitness(vec_to_coords(xx)) for xx in pbest_x])

    idxs = np.arange(particles)
    left = (idxs - 1) % particles; right = (idxs + 1) % particles
    def inertia_weight(t): return PSO_W_START + (PSO_W_END - PSO_W_START) * (t / max(1, iters-1))

    for t in range(iters):
        lbest = []
        for i in range(particles):
            neighbors = [i, left[i], right[i]]
            j = neighbors[np.argmax(pbest_f[neighbors])]
            lbest.append(pbest_x[j])
        lbest = np.array(lbest)

        w  = inertia_weight(t)
        r1 = rng.random(size=swarm_x.shape)
        r2 = rng.random(size=swarm_x.shape)
        swarm_v = (w*swarm_v + PSO_C1*r1*(pbest_x - swarm_x) + PSO_C2*r2*(lbest - swarm_x))
        swarm_x = clamp_vec(swarm_x + swarm_v)

        fvals = np.array([fitness(vec_to_coords(xx)) for xx in swarm_x])
        improved = fvals > pbest_f
        pbest_x[improved] = swarm_x[improved]
        pbest_f[improved] = fvals[improved]

    j = int(np.argmax(pbest_f))
    best = vec_to_coords(pbest_x[j])
    best, ok, _ = repair_route_to_sea(best, max_inserts=64)
    return best

# ============================================================
# 1-opt 安全簡化：嘗試刪除每個內點，只要不穿陸且距離更短就接受
# ============================================================
def simplify_one_opt(coords):
    changed = True
    r = coords[:]
    while changed:
        changed = False
        i = 1
        while i < len(r)-1 and len(r) > 2:
            cand = r[:i] + r[i+1:]
            if (not path_crosses_land(cand)) and (polyline_length_km(cand) + 1e-6 < polyline_length_km(r)):
                r = cand; changed = True
            else:
                i += 1
    return r

# ============================================================
# GA outer loop
# ============================================================
def tournament_select(pop, fits, k=3):
    idxs = np.random.choice(len(pop), size=k, replace=False)
    best = idxs[0]
    for i in idxs[1:]:
        if fits[i] > fits[best]: best = i
    return pop[best]

def crossover(p1, p2):
    m1, m2 = p1[1:-1], p2[1:-1]
    if not m1: return p1[:]
    if np.random.rand() < 0.5:
        cut = np.random.randint(1, len(m1))
        child_mid = m1[:cut] + m2[cut:]
    else:
        child_mid = []
        for a,b in zip(m1,m2):
            alpha = np.random.uniform(-0.2, 1.2)
            child_mid.append((a[0] + alpha*(b[0]-a[0]), a[1] + alpha*(b[1]-a[1])))
    child = [p1[0]] + child_mid + [p1[-1]]
    child, ok, _ = repair_route_to_sea(child, max_inserts=64)
    return child if ok else p1[:]

def mutate(ind, rate=GA_MUTATION_RATE):
    mids = ind[1:-1]
    if not mids: return ind[:]
    x0,y0,x1,y1 = AOI
    lon_scale = (x1-x0)*0.01; lat_scale=(y1-y0)*0.01
    mids2 = []
    for pt in mids:
        if np.random.rand() < rate:
            pt = (pt[0] + np.random.normal(0, lon_scale),
                  pt[1] + np.random.normal(0, lat_scale))
        mids2.append(pt)
    cand = [ind[0]] + mids2 + [ind[-1]]
    cand, ok, _ = repair_route_to_sea(cand, max_inserts=64)
    return cand if ok else ind[:]

def evo_swarm_optimize():
    pop = init_population_scgraph_only(GA_POP, K_WAYPOINTS)
    fits = np.array([fitness(ind) for ind in pop])
    best = pop[int(np.argmax(fits))]

    for g in range(GA_GENS):
        new_pop = []
        # elitism
        for i in np.argsort(-fits)[:GA_ELITISM]:
            new_pop.append(pop[i])

        while len(new_pop) < GA_POP:
            p1 = tournament_select(pop, fits, k=3)
            p2 = tournament_select(pop, fits, k=3)
            child = crossover(p1, p2) if np.random.rand() < GA_CROSSOVER_RATE else p1[:]
            child = mutate(child, rate=GA_MUTATION_RATE)
            child = pso_refine(child, iters=PSO_ITERS, particles=PSO_PARTICLES)
            child = simplify_one_opt(child)
            new_pop.append(child)

        pop  = new_pop
        fits = np.array([fitness(ind) for ind in pop])
        idx = int(np.argmax(fits))
        if fits[idx] > fitness(best):
            best = pop[idx]
        print(f"Gen {g+1:03d} | best_dist_km={polyline_length_km(best):.2f} | feasible={'✓' if not path_crosses_land(best) else '✗'}")
    return best

# ============================================================
# Build AOI / land / endpoints
# ============================================================
raw_start = (START_LON, START_LAT)
raw_end   = (END_LON,   END_LAT)
AOI = compute_aoi(raw_start, raw_end, AOI_PADDING_DEG)

# raw land for endpoint push-out
raw_polys, _ = _load_land_raw(AOI)

s0 = project_endpoint_to_sea_unbuffered(raw_start, raw_polys)
e0 = project_endpoint_to_sea_unbuffered(raw_end,   raw_polys)
# 再由岸邊往外推 X 公里，避免太貼岸
start = push_out_from_coast(s0, ENDPOINT_PUSH_OUT_KM, raw_polys)
end   = push_out_from_coast(e0, ENDPOINT_PUSH_OUT_KM, raw_polys)

# buffered land & prepared union for fast intersects
land_polys, land_prepared = _load_land_prepared(AOI)
print(f"Loaded {len(land_polys)} buffered land polygons; prepared union ready.")

# ============================================================
# Run
# ============================================================
print("\n" + "="*60)
print("  Evo+Swarm 海上路徑規劃器（scgraph-only / 無走廊吸附 / 安全簡化）")
print("="*60)

best_route = evo_swarm_optimize()
best_route = simplify_one_opt(best_route)   # 再簡化一次
best_dist  = polyline_length_km(best_route)
best_ok    = not path_crosses_land(best_route)

print("\n" + "="*60)
print("  最終結果")
print("="*60)
print(f"最佳距離: {best_dist:.2f} km")
print(f"是否可行: {'✓ 是' if best_ok else '✗ 否（仍穿陸）'}")
print(f"航點數量: {len(best_route)} 個")
print("="*60 + "\n")

# ============================================================
# Save outputs + Folium map
# ============================================================
geo = {
    "type": "FeatureCollection",
    "features": [{
        "type": "Feature",
        "properties": {"distance_km": best_dist, "feasible": best_ok, "waypoints": len(best_route)},
        "geometry": {"type": "LineString","coordinates": [[lon, lat] for (lon, lat) in best_route]}
    }]
}
os.makedirs(OUTPUT_DIR, exist_ok=True)
with open(os.path.join(OUTPUT_DIR, GEOJSON_NAME), "w", encoding="utf-8") as f:
    json.dump(geo, f, ensure_ascii=False, indent=2)
with open(os.path.join(OUTPUT_DIR, CSV_NAME), "w", encoding="utf-8") as f:
    f.write("lon,lat\n"); [f.write(f"{lon},{lat}\n") for lon,lat in best_route]
print("✓ Saved GeoJSON/CSV")

import folium
m = folium.Map(location=[(START_LAT+END_LAT)/2, (START_LON+END_LON)/2], zoom_start=5, tiles="cartodbpositron")
# 直線基準（原始設定點）
folium.PolyLine([(START_LAT, START_LON), (END_LAT, END_LON)], color="#ef4444", weight=2, opacity=0.6,
                tooltip="S→T straight").add_to(m)
# 最佳路徑
folium.PolyLine([(lat, lon) for (lon,lat) in best_route], color="#10b981" if best_ok else "#ef4444",
                weight=4, opacity=0.95, tooltip=f"Best: {best_dist:.2f} km").add_to(m)
# 端點 marker
folium.Marker(location=[START_LAT, START_LON], popup="Raw Start", icon=folium.Icon(color="lightgreen")).add_to(m)
folium.Marker(location=[best_route[0][1], best_route[0][0]], popup="Effective Start", icon=folium.Icon(color="green")).add_to(m)
folium.Marker(location=[END_LAT, END_LON], popup="Raw End", icon=folium.Icon(color="lightred")).add_to(m)
folium.Marker(location=[best_route[-1][1], best_route[-1][0]], popup="Effective End", icon=folium.Icon(color="red")).add_to(m)
m.save(os.path.join(OUTPUT_DIR, HTML_NAME))
print(f"✓ Saved map: {os.path.join(OUTPUT_DIR, HTML_NAME)}")
try:
    webbrowser.open(f"file:///{os.path.abspath(os.path.join(OUTPUT_DIR, HTML_NAME))}")
except Exception:
    pass
print("完成。")


Loaded 544 buffered land polygons; prepared union ready.

  Evo+Swarm 海上路徑規劃器（scgraph-only / 無走廊吸附 / 安全簡化）


RuntimeError: 仍無法建立可行種子；請放大 AOI 或檢查陸地底圖/緩衝。

In [ ]:
try:
    dbg = RSGB.sc_shortest_path_lonlat((START_LON,START_LAT),(END_LON,END_LAT),
                                       max_snap_km=50, inject_gateways=True, verbose=True)
    print("scgraph keys:", list(dbg.keys()))
    for k in ["snap_src_km","snap_dst_km","reason","n_nodes","n_edges"]:
        if k in dbg: print(k, "=", dbg[k])
except Exception as e:
    print("scgraph debug error:", e)

scgraph keys: ['track', 'length_km']


## End

# Evoswarm 

In [20]:
"""
routing_pipeline.py

簡化版海上路徑規劃 pipeline：

- 建立 RoutingContext（讀取陸地 shp、建投影與緩衝）
- port snapping: 若起訖點在陸地/緩衝上，推到海上
- repair_route: Kuhlemann 4.1 風格的幾何修補器
- simplify_route: 簡單的可視性簡化
- optimize_route: 很輕量的 evolutionary hill-climb 優化（只用 mutation）

依賴套件：
    pip install shapely pyproj fiona
"""

from __future__ import annotations
from dataclasses import dataclass
from typing import List, Tuple
import math
import random

from shapely.geometry import Point, LineString, Polygon, MultiPolygon, box
from shapely.ops import unary_union, transform
from shapely.strtree import STRtree
import fiona
from pyproj import CRS, Transformer
from shapely.geometry import shape
from shapely.geometry.base import BaseGeometry 
# -------------------------------
# 工具函式
# -------------------------------

LonLat = Tuple[float, float]
XY = Tuple[float, float]


def haversine_km(a: LonLat, b: LonLat) -> float:
    """球面距離（km），只用來回報長度，修補器內部用平面距離即可。"""
    lon1, lat1 = a
    lon2, lat2 = b
    R = 6371.0
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    s = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(
        dlambda / 2
    ) ** 2
    return 2 * R * math.asin(min(1.0, math.sqrt(s)))


def guess_utm_epsg(lon: float, lat: float) -> int:
    """根據一個點猜測適合的 UTM EPSG."""
    zone = int((lon + 180) // 6) + 1
    if lat >= 0:
        return 32600 + zone  # WGS84 / UTM 北半球
    else:
        return 32700 + zone  # 南半球


def as_multipolygon(geom) -> MultiPolygon:
    if isinstance(geom, MultiPolygon):
        return geom
    elif isinstance(geom, Polygon):
        return MultiPolygon([geom])
    else:
        # 可能是 GeometryCollection，取裡面 polygon
        polys = [g for g in geom.geoms if isinstance(g, (Polygon, MultiPolygon))]
        if len(polys) == 1 and isinstance(polys[0], MultiPolygon):
            return polys[0]
        return MultiPolygon([p for p in polys if isinstance(p, Polygon)])


# -------------------------------
# RoutingContext
# -------------------------------

@dataclass
class RoutingContext:
    to_xy: Transformer
    to_ll: Transformer
    land_buffer: MultiPolygon
    land_tree: STRtree
    roi_xy: Polygon
    seg_max_len_m: float
    push_dist_m: float


def build_routing_context(
    land_shp_path: str,
    roi_bbox_ll: Tuple[float, float, float, float],
    buffer_km: float = 5.0,
    seg_max_km: float = 80.0,
    push_km: float = 10.0,
) -> RoutingContext:
    """
    建立 RoutingContext。

    roi_bbox_ll: (min_lon, min_lat, max_lon, max_lat)
    """
    min_lon, min_lat, max_lon, max_lat = roi_bbox_ll

    # 選擇一個適合 ROI 的 UTM 投影
    mid_lon = (min_lon + max_lon) / 2
    mid_lat = (min_lat + max_lat) / 2
    xy_epsg = guess_utm_epsg(mid_lon, mid_lat)

    crs_ll = CRS.from_epsg(4326)
    crs_xy = CRS.from_epsg(xy_epsg)
    to_xy = Transformer.from_crs(crs_ll, crs_xy, always_xy=True)
    to_ll = Transformer.from_crs(crs_xy, crs_ll, always_xy=True)

    # ROI 轉到投影座標
    x1, y1 = to_xy.transform(min_lon, min_lat)
    x2, y2 = to_xy.transform(max_lon, max_lat)
    roi_xy = box(min(x1, x2), min(y1, y2), max(x1, x2), max(y1, y2))

    # 讀取 land shp，轉成 xy，裁切 ROI
    land_geoms_xy: List[Polygon] = []
    from shapely.geometry import shape  # 可以放在檔案開頭 import，一次就好

    from shapely.geometry import shape
    from shapely.errors import GEOSException  # 新增
# ...

    # 讀取 land shp，轉成 xy，裁切 ROI
    land_geoms_xy: List[Polygon] = []

    with fiona.open(land_shp_path) as src:
        for feat in src:
            g_ll = feat["geometry"]
            if g_ll is None:
                continue

            # 1) GeoJSON dict -> shapely geometry（經緯度座標）
            geom_ll = shape(g_ll)

            # 1.1 先修一下 invalid 幾何（很多 Natural Earth 會有）
            if not geom_ll.is_valid:
                geom_ll = geom_ll.buffer(0)
                if geom_ll.is_empty:
                    continue

            # 2) 先在經緯度空間跟 ROI（經緯度 box）裁切
            roi_ll_box = box(min_lon, min_lat, max_lon, max_lat)
            try:
                clipped_ll = geom_ll.intersection(roi_ll_box)
            except GEOSException as e:
                print(f"[WARN] intersection in lon/lat failed, skipping feature: {e}")
                continue

            if clipped_ll.is_empty:
                continue

            # 3) 再轉成投影座標
            geom_xy = transform(to_xy.transform, clipped_ll)

            # 3.1 再保險一次 invalid 修正
            if not geom_xy.is_valid:
                geom_xy = geom_xy.buffer(0)
                if geom_xy.is_empty:
                    continue

            # 4) 最後再跟 roi_xy 做一次交集（通常這步其實已經不太必要，但保險）
            try:
                clipped = geom_xy.intersection(roi_xy)
            except GEOSException as e:
                print(f"[WARN] intersection in XY failed, skipping feature: {e}")
                continue

            if not clipped.is_empty:
                land_geoms_xy.append(clipped)



    if not land_geoms_xy:
        raise ValueError("ROI 裡面沒有讀到任何陸地，請確認 shp 跟 ROI。")

    land_union = unary_union(land_geoms_xy)
    land_buffer = as_multipolygon(
        land_union.buffer(buffer_km * 1000.0, cap_style=1, join_style=1)
    )

    geom_list = list(land_buffer.geoms)
    land_tree = STRtree(geom_list)

    ctx = RoutingContext(
        to_xy=to_xy,
        to_ll=to_ll,
        land_buffer=land_buffer,
        land_tree=land_tree,
        roi_xy=roi_xy,
        seg_max_len_m=seg_max_km * 1000.0,
        push_dist_m=push_km * 1000.0,
    )
    return ctx


# -------------------------------
# 幾何判定
# -------------------------------


def point_in_land(ctx: RoutingContext, p_xy: XY) -> bool:
    """
    檢查點是否在陸地 buffer 裡（支援 Shapely 2.x 的 STRtree）
    """
    pt = Point(p_xy)
    # STRtree.query 在 Shapely 2.x 回傳的是「index」
    idxs = ctx.land_tree.query(pt)  # ndarray of indices

    if idxs is None or len(idxs) == 0:
        return False

    geoms = ctx.land_tree.geometries  # numpy array of geometries
    for i in idxs:
        g = geoms[i]
        # 保險一下：只處理 geometry 物件
        if isinstance(g, BaseGeometry) and pt.within(g):
            return True
    return False



def segment_hits_land(ctx: RoutingContext, a_xy: XY, b_xy: XY, debug: bool = False) -> bool:
    """
    檢查線段是否穿過陸地（支援 Shapely 2.x 的 STRtree）
    """
    line = LineString([a_xy, b_xy])
    
    # query 回來的是「陸地 polygon 的 index」
    idxs = ctx.land_tree.query(line)

    if debug:
        print(f"  Line from {a_xy} to {b_xy}")
        print(f"  Line length: {line.length:.1f}m")
        print(f"  Candidates from STRtree: {0 if idxs is None else len(idxs)}")

    if idxs is None or len(idxs) == 0:
        if debug:
            print("  No candidates found - line doesn't hit land")
        return False

    geoms = ctx.land_tree.geometries

    # 再用真正的幾何做精查
    for idx in idxs:
        g = geoms[idx]
        if line.intersects(g):
            if debug:
                print(f"  HIT! Intersects with polygon index {idx}")
            return True

    if debug:
        print("  No actual intersection despite candidates")

    return False


def test_direct_path(ctx: RoutingContext, start_ll: LonLat, end_ll: LonLat):
    """
    測試函數：檢查直線路徑是否穿陸
    """
    print(f"\n=== Testing direct path ===")
    print(f"Start: {start_ll}")
    print(f"End: {end_ll}")
    
    sx, sy = ctx.to_xy.transform(start_ll[0], start_ll[1])
    ex, ey = ctx.to_xy.transform(end_ll[0], end_ll[1])
    
    print(f"Start XY: ({sx:.1f}, {sy:.1f})")
    print(f"End XY: ({ex:.1f}, {ey:.1f})")
    print(f"Distance: {segment_len_m((sx, sy), (ex, ey))/1000:.1f} km")
    
    # 檢查起終點是否在陸地上
    print(f"\nStart point in land: {point_in_land(ctx, (sx, sy))}")
    print(f"End point in land: {point_in_land(ctx, (ex, ey))}")
    
    # 檢查直線是否穿陸
    print(f"\nDirect segment hits land:")
    hits = segment_hits_land(ctx, (sx, sy), (ex, ey), debug=True)
    print(f"Result: {hits}")
    
    # 檢查陸地多邊形數量
    print(f"\nTotal land polygons in buffer: {len(list(ctx.land_buffer.geoms))}")
    
    return hits


def segment_len_m(a_xy: XY, b_xy: XY) -> float:
    dx = b_xy[0] - a_xy[0]
    dy = b_xy[1] - a_xy[1]
    return math.hypot(dx, dy)


# -------------------------------
# Port snapping
# -------------------------------

def snap_port_to_sea(ctx: RoutingContext, lon: float, lat: float, debug: bool = False) -> LonLat:
    """
    若港口在陸地/緩衝內，往外推到海上。
    修正版：增加更多搜索方向和距離，並加入除錯輸出
    """
    x, y = ctx.to_xy.transform(lon, lat)
    p_xy = (x, y)
    
    if debug:
        print(f"\n=== Snapping port ({lon:.4f}, {lat:.4f}) ===")
        print(f"XY: ({x:.1f}, {y:.1f})")
    
    # 檢查是否在陸地上
    in_land = point_in_land(ctx, p_xy)
    if debug:
        print(f"Point in land: {in_land}")
    
    if not in_land:
        if debug:
            print(f"Already in sea, no snapping needed")
        return lon, lat

    # 在陸地上，需要往外推
    if debug:
        print(f"Port is on land, searching for sea...")
    
    # 增加搜索密度和範圍
    angles = range(0, 360, 10)  # 每 10 度一個方向
    max_steps = 20  # 最多推 20 步
    
    for step in range(1, max_steps + 1):
        for angle_deg in angles:
            angle = math.radians(angle_deg)
            dx = math.cos(angle)
            dy = math.sin(angle)
            
            # 每次推 push_dist_m 的距離
            cand_x = x + dx * ctx.push_dist_m * step
            cand_y = y + dy * ctx.push_dist_m * step
            cand = (cand_x, cand_y)
            
            # 檢查候選點是否在海上且在 ROI 內
            if not point_in_land(ctx, cand) and ctx.roi_xy.contains(Point(cand)):
                lon2, lat2 = ctx.to_ll.transform(cand[0], cand[1])
                if debug:
                    distance_km = ctx.push_dist_m * step / 1000.0
                    print(f"Found sea at angle {angle_deg}°, distance {distance_km:.1f}km")
                    print(f"New position: ({lon2:.4f}, {lat2:.4f})")
                return lon2, lat2
    
    # 實在找不到，回傳原座標並警告
    if debug:
        print(f"WARNING: Could not find sea within {max_steps * ctx.push_dist_m / 1000:.1f}km!")
        print(f"Returning original coordinates")
    
    return lon, lat


def visualize_port_snapping(
    ctx: RoutingContext,
    port_ll: LonLat,
    port_name: str = "Port",
    filename: str = "port_snapping.html"
):
    """
    視覺化港口 snapping 的過程
    """
    import folium
    import webbrowser
    import os
    
    # 執行 snapping
    print(f"\nSnapping {port_name}...")
    snapped_ll = snap_port_to_sea(ctx, port_ll[0], port_ll[1], debug=True)
    
    # 創建地圖（以原始港口為中心）
    m = folium.Map(
        location=[port_ll[1], port_ll[0]],
        zoom_start=8,
        tiles='OpenStreetMap'
    )
    
    # 畫陸地緩衝區
    for poly in ctx.land_buffer.geoms:
        poly_ll = transform(ctx.to_ll.transform, poly)
        coords = list(poly_ll.exterior.coords)
        coords_latlon = [(lat, lon) for lon, lat in coords]
        
        folium.Polygon(
            locations=coords_latlon,
            color='blue',
            fill=True,
            fillColor='lightblue',
            fillOpacity=0.4,
            weight=2,
            popup='Land Buffer'
        ).add_to(m)
    
    # 標記原始港口位置（紅色）
    folium.Marker(
        location=[port_ll[1], port_ll[0]],
        popup=f'Original {port_name}<br>({port_ll[0]:.4f}, {port_ll[1]:.4f})<br>(On Land)',
        icon=folium.Icon(color='red', icon='anchor', prefix='fa')
    ).add_to(m)
    
    # 標記 snapped 後的位置（綠色）
    if snapped_ll != port_ll:
        folium.Marker(
            location=[snapped_ll[1], snapped_ll[0]],
            popup=f'Snapped {port_name}<br>({snapped_ll[0]:.4f}, {snapped_ll[1]:.4f})<br>(In Sea)',
            icon=folium.Icon(color='green', icon='anchor', prefix='fa')
        ).add_to(m)
        
        # 畫連線
        folium.PolyLine(
            locations=[(port_ll[1], port_ll[0]), (snapped_ll[1], snapped_ll[0])],
            color='orange',
            weight=3,
            opacity=0.8,
            dash_array='5, 5',
            popup=f'Snapping distance: {haversine_km(port_ll, snapped_ll):.1f} km'
        ).add_to(m)
    else:
        # 沒有 snapping（已經在海上）
        folium.Marker(
            location=[port_ll[1], port_ll[0]],
            popup=f'{port_name}<br>Already in sea',
            icon=folium.Icon(color='green', icon='check')
        ).add_to(m)
    
    # 儲存並開啟
    m.save(filename)
    abs_path = os.path.abspath(filename)
    webbrowser.open('file://' + abs_path)
    print(f"Port snapping map saved to {filename}")
    
    return snapped_ll


# -------------------------------
# 修補器：Kuhlemann 4.1 風格
# -------------------------------
def repair_route_xy(ctx: RoutingContext, pts_xy: List[XY]) -> List[XY]:
    """
    給一串 xy 點（至少兩個），回傳修補後的 route（不穿陸、段長合理）。
    修正版：避免重複添加點
    """
    route = pts_xy[:]
    assert len(route) >= 2

    changed = True
    max_iter = 1000
    it = 0
    while changed and it < max_iter:
        it += 1
        changed = False
        new_route: List[XY] = []
        
        for i in range(len(route) - 1):
            a = route[i]
            b = route[i + 1]
            
            # 添加起點（只在第一次或前一段已經處理完時添加）
            if i == 0:
                new_route.append(a)
            
            need_split = False
            if segment_len_m(a, b) > ctx.seg_max_len_m:
                need_split = True
            if segment_hits_land(ctx, a, b):
                need_split = True

            if need_split:
                # 找中點
                mx = (a[0] + b[0]) / 2.0
                my = (a[1] + b[1]) / 2.0
                mid = (mx, my)
                
                # 若中點在 land_buffer，就往法向量方向外推
                if point_in_land(ctx, mid):
                    dx = b[0] - a[0]
                    dy = b[1] - a[1]
                    if dx == 0 and dy == 0:
                        dx, dy = 1.0, 0.0
                    nx, ny = -dy, dx
                    nlen = math.hypot(nx, ny)
                    nx /= nlen
                    ny /= nlen

                    # 兩側都試試看
                    moved = None
                    for sign in (+1, -1):
                        cand = (
                            mx + sign * nx * ctx.push_dist_m,
                            my + sign * ny * ctx.push_dist_m,
                        )
                        if not point_in_land(ctx, cand):
                            moved = cand
                            break
                    if moved is None:
                        # 兩側都沒成功，就多推幾步
                        for sign in (+1, -1):
                            for k in range(2, 6):
                                cand = (
                                    mx + sign * nx * ctx.push_dist_m * k,
                                    my + sign * ny * ctx.push_dist_m * k,
                                )
                                if not point_in_land(ctx, cand):
                                    moved = cand
                                    break
                            if moved is not None:
                                break
                    mid = moved if moved is not None else mid

                new_route.append(mid)
                changed = True
            
            # 添加終點
            new_route.append(b)

        route = new_route

    return route



def repair_route_ll(ctx: RoutingContext, start_ll: LonLat, end_ll: LonLat) -> List[LonLat]:
    """封裝版：輸入 lon/lat 起訖，輸出修補後的 lon/lat polyline。"""
    sx, sy = ctx.to_xy.transform(start_ll[0], start_ll[1])
    ex, ey = ctx.to_xy.transform(end_ll[0], end_ll[1])
    route_xy = repair_route_xy(ctx, [(sx, sy), (ex, ey)])
    route_ll: List[LonLat] = [
        ctx.to_ll.transform(x, y) for x, y in route_xy
    ]
    return route_ll


# -------------------------------
# 第一輪簡化
# -------------------------------

def simplify_route_ll(ctx: RoutingContext, route_ll: List[LonLat], tol_km: float = 5.0, max_segment_km: float = 200.0) -> List[LonLat]:
    """
    修正版：改進簡化邏輯，避免過度簡化
    新增 max_segment_km 限制，避免簡化後的段落太長
    """
    if len(route_ll) <= 2:
        return route_ll[:]

    # 先轉 xy
    route_xy: List[XY] = [ctx.to_xy.transform(lon, lat) for lon, lat in route_ll]
    tol_m = tol_km * 1000.0
    max_seg_m = max_segment_km * 1000.0

    simplified_xy: List[XY] = [route_xy[0]]
    i = 0
    n = len(route_xy)
    
    while i < n - 1:
        j = i + 1
        last_good = i
        
        # 逐步延伸，找到最遠的可達點
        while j < n:
            a = route_xy[i]
            b = route_xy[j]
            
            # 檢查段長度是否超過限制
            seg_len = segment_len_m(a, b)
            if seg_len > max_seg_m:
                break
            
            # 檢查是否穿陸
            if segment_hits_land(ctx, a, b):
                break

            # 偏離檢查：中間點到直線距離
            line = LineString([a, b])
            max_dev = 0.0
            for k in range(i + 1, j):
                p = Point(route_xy[k])
                dev = p.distance(line)
                max_dev = max(max_dev, dev)
            
            if max_dev > tol_m:
                break

            last_good = j
            j += 1

        # 如果沒有進展（last_good == i），至少前進一步
        if last_good == i:
            last_good = i + 1
        
        # 添加找到的點（除非是最後一個點）
        if last_good < n - 1:
            simplified_xy.append(route_xy[last_good])
        
        i = last_good

    simplified_xy.append(route_xy[-1])

    return [ctx.to_ll.transform(x, y) for x, y in simplified_xy]


# -------------------------------
# 非常輕量的優化器（mutation + hill-climb）
# -------------------------------

def route_length_km(route_ll: List[LonLat]) -> float:
    return sum(haversine_km(route_ll[i], route_ll[i + 1]) for i in range(len(route_ll) - 1))


def mutate_delete_arc(ctx: RoutingContext, route_ll: List[LonLat]) -> List[LonLat]:
    if len(route_ll) <= 3:
        return route_ll[:]
    n = len(route_ll)
    # 隨機選一段中間 arc
    i = random.randint(0, n - 3)
    j = random.randint(i + 2, n - 1)
    start = route_ll[i]
    end = route_ll[j]
    # 用修補器重接這段
    sub_route = repair_route_ll(ctx, start, end)
    new_route = route_ll[: i + 1] + sub_route[1:-1] + route_ll[j:]
    return new_route


def mutate_move_point(ctx: RoutingContext, route_ll: List[LonLat], move_km: float = 20.0) -> List[LonLat]:
    """
    修正版：只修補受影響的段落
    """
    if len(route_ll) <= 2:
        return route_ll[:]

    idx = random.randint(1, len(route_ll) - 2)  # 不動起訖
    lon, lat = route_ll[idx]
    x, y = ctx.to_xy.transform(lon, lat)

    # 隨機方向移動一小段
    angle = random.random() * 2 * math.pi
    dist_m = move_km * 1000.0 * random.random()
    nx = x + math.cos(angle) * dist_m
    ny = y + math.sin(angle) * dist_m

    new_lon, new_lat = ctx.to_ll.transform(nx, ny)
    
    # 只修補前後兩段
    before = route_ll[:idx]
    after = route_ll[idx+1:]
    
    # 修補 [idx-1] -> new_point
    seg1 = repair_route_ll(ctx, route_ll[idx-1], (new_lon, new_lat))
    # 修補 new_point -> [idx+1]
    seg2 = repair_route_ll(ctx, (new_lon, new_lat), route_ll[idx+1])
    
    # 組合（避免重複點）
    new_route = before + seg1[:-1] + seg2 + after
    
    return new_route


def optimize_route(
    ctx: RoutingContext,
    route_ll: List[LonLat],
    n_iters: int = 200,
) -> List[LonLat]:
    """
    超簡化版 evolutionary hill-climb：
    - 只有 delete_arc / move_point 兩種 mutation
    - 每次只接受更短的解
    """
    best = simplify_route_ll(ctx, route_ll)
    best_len = route_length_km(best)

    for it in range(n_iters):
        if random.random() < 0.5:
            cand = mutate_delete_arc(ctx, best)
        else:
            cand = mutate_move_point(ctx, best)

        cand = simplify_route_ll(ctx, cand)
        cand_len = route_length_km(cand)
        if cand_len + 1e-6 < best_len:
            best, best_len = cand, cand_len
            # print(f"iter {it}: improved to {best_len:.1f} km")

    return best


# -------------------------------
# 對外主要 API
# -------------------------------

def plan_route(
    ctx: RoutingContext,
    start_ll: LonLat,
    end_ll: LonLat,
    n_opt_iters: int = 200,
) -> List[LonLat]:
    # 1) 港口 snapping
    s_snapped = snap_port_to_sea(ctx, *start_ll)
    e_snapped = snap_port_to_sea(ctx, *end_ll)
    print(f"Snapped: {s_snapped} -> {e_snapped}")

    # 2) 初始航線（修補器）
    base_route = repair_route_ll(ctx, s_snapped, e_snapped)
    print(f"After repair: {len(base_route)} points, {route_length_km(base_route):.1f} km")

    # 3) 第一輪簡化
    base_route = simplify_route_ll(ctx, base_route)
    print(f"After simplify: {len(base_route)} points, {route_length_km(base_route):.1f} km")

    # 4) 輕量優化
    if n_opt_iters > 0:
        best = optimize_route(ctx, base_route, n_iters=n_opt_iters)
        print(f"After optimize: {len(best)} points, {route_length_km(best):.1f} km")
    else:
        best = base_route

    return best



## 視覺化工具

In [21]:
"""
路徑視覺化工具
pip install folium
"""
import folium
import webbrowser
import os
from typing import List
from shapely.geometry import mapping


def visualize_route(
    ctx: RoutingContext,
    route_ll: List[LonLat],
    start_ll: LonLat,
    end_ll: LonLat,
    filename: str = "route_map.html"
):
    """
    用 folium 視覺化路徑
    """
    # 計算地圖中心
    center_lat = (start_ll[1] + end_ll[1]) / 2
    center_lon = (start_ll[0] + end_ll[0]) / 2
    
    # 創建地圖
    m = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=5,
        tiles='OpenStreetMap'
    )
    
    # 1. 畫陸地緩衝區（藍色半透明）
    print("Drawing land buffer...")
    for poly in ctx.land_buffer.geoms:
        # 轉回經緯度
        poly_ll = transform(ctx.to_ll.transform, poly)
        coords = list(poly_ll.exterior.coords)
        coords_latlon = [(lat, lon) for lon, lat in coords]
        
        folium.Polygon(
            locations=coords_latlon,
            color='blue',
            fill=True,
            fillColor='lightblue',
            fillOpacity=0.3,
            weight=1,
            popup='Land Buffer (5km)'
        ).add_to(m)
    
    # 2. 畫路徑（紅色粗線）
    if len(route_ll) >= 2:
        route_latlon = [(lat, lon) for lon, lat in route_ll]
        folium.PolyLine(
            locations=route_latlon,
            color='red',
            weight=3,
            opacity=0.8,
            popup=f'Route: {len(route_ll)} points, {route_length_km(route_ll):.1f} km'
        ).add_to(m)
    
    # 3. 畫航點（小圓點）
    for i, (lon, lat) in enumerate(route_ll):
        color = 'green' if i == 0 else ('red' if i == len(route_ll)-1 else 'orange')
        label = 'Start' if i == 0 else ('End' if i == len(route_ll)-1 else f'Point {i}')
        
        folium.CircleMarker(
            location=[lat, lon],
            radius=5 if i == 0 or i == len(route_ll)-1 else 3,
            color=color,
            fill=True,
            fillColor=color,
            fillOpacity=0.8,
            popup=f'{label}: ({lon:.4f}, {lat:.4f})'
        ).add_to(m)
    
    # 4. 畫起點到終點的直線（虛線，用來對比）
    folium.PolyLine(
        locations=[(start_ll[1], start_ll[0]), (end_ll[1], end_ll[0])],
        color='purple',
        weight=2,
        opacity=0.5,
        dash_array='10, 10',
        popup=f'Direct line: {haversine_km(start_ll, end_ll):.1f} km'
    ).add_to(m)
    
    # 5. 標記起終點
    folium.Marker(
        location=[start_ll[1], start_ll[0]],
        popup=f'Start: Perth<br>({start_ll[0]:.4f}, {start_ll[1]:.4f})',
        icon=folium.Icon(color='green', icon='play')
    ).add_to(m)
    
    folium.Marker(
        location=[end_ll[1], end_ll[0]],
        popup=f'End: Brisbane<br>({end_ll[0]:.4f}, {end_ll[1]:.4f})',
        icon=folium.Icon(color='red', icon='stop')
    ).add_to(m)
    
    # 加入圖例
    legend_html = '''
    <div style="position: fixed; 
                bottom: 50px; right: 50px; width: 200px; height: 180px; 
                background-color: white; border:2px solid grey; z-index:9999; 
                font-size:14px; padding: 10px">
    <p><strong>Legend</strong></p>
    <p><span style="color:lightblue">█</span> Land Buffer (5km)</p>
    <p><span style="color:red">━━━</span> Computed Route</p>
    <p><span style="color:purple">┉┉┉</span> Direct Line</p>
    <p><span style="color:green">●</span> Start Port</p>
    <p><span style="color:red">●</span> End Port</p>
    <p><span style="color:orange">●</span> Waypoints</p>
    </div>
    '''
    m.get_root().html.add_child(folium.Element(legend_html))
    
    # 儲存並開啟
    m.save(filename)
    print(f"Map saved to {filename}")
    
    # 在瀏覽器中打開
    abs_path = os.path.abspath(filename)
    webbrowser.open('file://' + abs_path)
    print(f"Opening in browser...")


def visualize_repair_steps(
    ctx: RoutingContext,
    start_ll: LonLat,
    end_ll: LonLat,
    filename: str = "repair_steps.html"
):
    """
    視覺化修補過程的步驟
    """
    # 計算中心
    center_lat = (start_ll[1] + end_ll[1]) / 2
    center_lon = (start_ll[0] + end_ll[0]) / 2
    
    m = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=5,
        tiles='OpenStreetMap'
    )
    
    # 畫陸地
    for poly in ctx.land_buffer.geoms:
        poly_ll = transform(ctx.to_ll.transform, poly)
        coords = list(poly_ll.exterior.coords)
        coords_latlon = [(lat, lon) for lon, lat in coords]
        
        folium.Polygon(
            locations=coords_latlon,
            color='blue',
            fill=True,
            fillColor='lightblue',
            fillOpacity=0.3,
            weight=1
        ).add_to(m)
    
    # 先做 snapping
    s_snapped = snap_port_to_sea(ctx, *start_ll)
    e_snapped = snap_port_to_sea(ctx, *end_ll)
    
    # 執行修補並記錄每個點
    sx, sy = ctx.to_xy.transform(s_snapped[0], s_snapped[1])
    ex, ey = ctx.to_xy.transform(e_snapped[0], e_snapped[1])
    route_xy = repair_route_xy(ctx, [(sx, sy), (ex, ey)])
    
    # 轉回經緯度
    route_ll = [ctx.to_ll.transform(x, y) for x, y in route_xy]
    
    # 畫修補後的路徑
    if len(route_ll) >= 2:
        route_latlon = [(lat, lon) for lon, lat in route_ll]
        folium.PolyLine(
            locations=route_latlon,
            color='red',
            weight=3,
            opacity=0.8,
            popup=f'Repaired Route: {len(route_ll)} points'
        ).add_to(m)
    
    # 標記每個點
    for i, (lon, lat) in enumerate(route_ll):
        color = 'green' if i == 0 else ('red' if i == len(route_ll)-1 else 'blue')
        
        folium.CircleMarker(
            location=[lat, lon],
            radius=4,
            color=color,
            fill=True,
            fillColor=color,
            fillOpacity=0.7,
            popup=f'Point {i}: ({lon:.4f}, {lat:.4f})'
        ).add_to(m)
    
    # 畫直線對比
    folium.PolyLine(
        locations=[(s_snapped[1], s_snapped[0]), (e_snapped[1], e_snapped[0])],
        color='purple',
        weight=2,
        opacity=0.5,
        dash_array='10, 10',
        popup='Direct line (would hit land)'
    ).add_to(m)
    
    m.save(filename)
    abs_path = os.path.abspath(filename)
    webbrowser.open('file://' + abs_path)
    print(f"Repair steps map saved to {filename}")

## Main testing

In [23]:
if __name__ == "__main__":
    # 你的主程式
    land_shp = r"C:\Users\slab\Desktop\Slab Project\Stage1\data\Land\ne_10m_land.shp"
    roi = (100.0, 5.0, 125.0, 26.0)

    ctx = build_routing_context(
        land_shp_path=land_shp,
        roi_bbox_ll=roi,
        buffer_km=5.0,
        seg_max_km=80.0,
        push_km=15.0,
    )

    # Perth 和 Brisbane 的實際座標（在陸地上）
    start = (120.3181,22.58425) 
    end = (108.647, 21.68317)     
    
    #| KAOHSIUNG  | TWKHH     | (22.58425,120.3181)  | (120.3181,22.58425)            |
    #| TAICHUNG   | TWTXG     | (24.255,120.5075)    | (120.5075,24.255)              |
    #| KEELUNG    | TWKEL     | (25.14515,121.7532)  | (121.7532,25.14515)            |
    #| SHANGHAI   | CNSHG     | (31.36636,121.6147)  | (121.6147,31.36636)            |
    #| NINGBO     | CNNBG     | (29.92654,121.8525)  | (121.8525,29.92654)            |
    #| ZHOUSHAN   | CNZOS     | (29.92161,122.2104)  | (122.2104,29.92161)            |
    #| SHENZHEN   | CNSZX     | (22.5045,113.8535)   | (113.8535,22.5045)             |
    #| LOS ANGELES| USLAX     | (33.74021,-118.265)  | (-118.265,33.74021)            |
    #| SEATTLE    | USSEA     | (47.6212,-122.3643)  | (-122.3643,47.6212)            |
    #| TOKYO      | JPTYO     | (35.61168,139.8268)  | (139.8268,35.61168)            |
    #| KOBE       | JPUKB     | (34.6867,135.2671)   | (135.2671,34.6867)             |
    #| WAKAYAMA   | JPUKW     | (33.468056,135.828056) | (135.828056,33.468056)       |
    #| BUSAN      | KRPUS     | (35.116667,129.033333) | (129.033333,35.116667)       |00
    #| Sihanoukville| KHKOS   | (10.633333,103.5)    | (103.5, 10.633333)             |
    #| QINZHOU    | CNQZH     | ----                 | (108.647, 21.68317)            |


    # 1. 先視覺化港口 snapping
    print("="*60)
    print("Step 1: Port Snapping")
    print("="*60)
    visualize_port_snapping(ctx, start, "Perth", "perth_snapping.html")
    visualize_port_snapping(ctx, end, "Brisbane", "brisbane_snapping.html")
    
    # 2. 視覺化修補步驟
    print("\n" + "="*60)
    print("Step 2: Route Repair")
    print("="*60)
    visualize_repair_steps(ctx, start, end, "repair_steps.html")
    
    # 3. 完整路徑規劃
    print("\n" + "="*60)
    print("Step 3: Full Route Planning")
    print("="*60)
    route = plan_route(ctx, start, end, n_opt_iters=300)
    
    print(f"\nFinal route: {len(route)} points, {route_length_km(route):.1f} km")
    
    # 4. 視覺化最終路徑
    print("\n" + "="*60)
    print("Step 4: Final Route Visualization")
    print("="*60)
    visualize_route(ctx, route, start, end, "final_route.html")
    
    print("\n" + "="*60)
    print("All visualizations complete!")
    print("="*60)
    print("Generated files:")
    print("  - perth_snapping.html")
    print("  - brisbane_snapping.html")
    print("  - repair_steps.html")
    print("  - final_route.html")

Step 1: Port Snapping

Snapping Perth...

=== Snapping port (120.3181, 22.5843) ===
XY: (1460837.9, 2527685.8)
Point in land: True
Port is on land, searching for sea...
Found sea at angle 160°, distance 15.0km
New position: (120.1858, 22.6379)
Port snapping map saved to perth_snapping.html

Snapping Brisbane...

=== Snapping port (108.6470, 21.6832) ===
XY: (256535.5, 2399606.8)
Point in land: True
Port is on land, searching for sea...
Found sea at angle 240°, distance 15.0km
New position: (108.5765, 21.5649)
Port snapping map saved to brisbane_snapping.html

Step 2: Route Repair


KeyboardInterrupt: 

## 檢查大陸塊

In [15]:
import fiona
import folium
import webbrowser
import tempfile
import os
import math

from shapely.geometry import shape, box, mapping
from shapely.ops import unary_union, transform
from pyproj import CRS, Transformer


# ======== 使用者設定 ========
land_shp = r"C:\Users\slab\Desktop\Slab Project\Stage1\data\Land\ne_10m_land.shp"
roi = (110.0, -45.0, 160.0, -5.0)   # (min_lon, min_lat, max_lon, max_lat)
buffer_km = 5.0
# ===========================


def guess_utm_epsg(lon, lat):
    """簡單猜一個適合 ROI 的 UTM EPSG。"""
    zone = int((lon + 180) // 6) + 1
    if lat >= 0:
        return 32600 + zone  # 北半球
    else:
        return 32700 + zone  # 南半球


min_lon, min_lat, max_lon, max_lat = roi
mid_lon = (min_lon + max_lon) / 2
mid_lat = (min_lat + max_lat) / 2

# 建立經緯度 <-> 投影座標轉換
crs_ll = CRS.from_epsg(4326)
crs_xy = CRS.from_epsg(guess_utm_epsg(mid_lon, mid_lat))
to_xy = Transformer.from_crs(crs_ll, crs_xy, always_xy=True)
to_ll = Transformer.from_crs(crs_xy, crs_ll, always_xy=True)

# ROI 轉成 polygon（先在經緯度空間，再轉投影）
roi_ll_box = box(min_lon, min_lat, max_lon, max_lat)
roi_xy_box = transform(to_xy.transform, roi_ll_box)

land_geoms_xy = []

print("讀取 land shapefile 並裁切 ROI ...")

with fiona.open(land_shp) as src:
    for feat in src:
        g_ll = feat["geometry"]
        if g_ll is None:
            continue

        geom_ll = shape(g_ll)                       # (lon, lat)
        geom_xy = transform(to_xy.transform, geom_ll)  # 轉投影座標
        clipped = geom_xy.intersection(roi_xy_box)  # 裁切 ROI

        if not clipped.is_empty:
            land_geoms_xy.append(clipped)

if not land_geoms_xy:
    print("⚠️ ROI 裡沒有裁到任何陸地，請檢查 ROI 或 shapefile。")
else:
    print(f"在 ROI 內找到 {len(land_geoms_xy)} 個陸地幾何。")

    # 合併所有陸地，再做 5 km buffer
    land_union_xy = unary_union(land_geoms_xy)
    buffer_m = buffer_km * 1000.0
    land_buffer_xy = land_union_xy.buffer(buffer_m, cap_style=1, join_style=1)

    # 轉回經緯度，用來給 folium 畫
    land_buffer_ll = transform(to_ll.transform, land_buffer_xy)

    # 建 folium 地圖
    center_lat = (min_lat + max_lat) / 2
    center_lon = (min_lon + max_lon) / 2
    m = folium.Map(location=[center_lat, center_lon], zoom_start=4, tiles="OpenStreetMap")

    # 畫 ROI 框（方便確認）
    folium.GeoJson(
        mapping(roi_ll_box),
        name="ROI",
        style_function=lambda x: {"color": "blue", "weight": 2, "fill": False},
    ).add_to(m)

    # 畫陸地 buffer（5km）
    folium.GeoJson(
        mapping(land_buffer_ll),
        name="land_buffer_5km",
        style_function=lambda x: {"color": "red", "weight": 1, "fill": True, "fillOpacity": 0.5},
    ).add_to(m)

    folium.LayerControl().add_to(m)

    # 存成暫存 html 並打開
    tmp_dir = tempfile.gettempdir()
    html_path = os.path.join(tmp_dir, "land_buffer_5km_roi.html")
    m.save(html_path)
    print(f"✅ 地圖已存成：{html_path}")
    webbrowser.open(f"file://{html_path}")


讀取 land shapefile 並裁切 ROI ...
在 ROI 內找到 6 個陸地幾何。
✅ 地圖已存成：C:\Users\slab\AppData\Local\Temp\land_buffer_5km_roi.html
